In [1]:
# ============================================================
# GPU STEP 1 — ENVIRONMENT + ASSET VERIFICATION
# ============================================================

from pathlib import Path
import json
import pandas as pd
import torch

print("=" * 80)
print("GPU STEP 1 — ENVIRONMENT + ASSET VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. GPU check
# ------------------------------------------------------------

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )
else:
    raise RuntimeError(
        "GPU is not enabled. Turn on a Kaggle GPU accelerator first."
    )


# ------------------------------------------------------------
# 2. Input roots
# ------------------------------------------------------------

COMP_ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

CPU_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-soft-label-production/"
    "soft_supervised_mri_cpu"
)

SSL_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-clean-ssl-training"
)

print("\nCompetition root exists:", COMP_ROOT.exists())
print("CPU root exists:", CPU_ROOT.exists())
print("SSL root exists:", SSL_ROOT.exists())


# ------------------------------------------------------------
# 3. Required files
# ------------------------------------------------------------

MANIFEST_PATH = (
    CPU_ROOT /
    "combined_4407_supervised_triplets.parquet"
)

PERCENTILES_PATH = (
    CPU_ROOT /
    "combined_24371_series_percentiles.parquet"
)

CONFIG_PATH = (
    CPU_ROOT /
    "training_config.json"
)

WEIGHTS_PATH = (
    CPU_ROOT /
    "target_supervision_weights.csv"
)

STUDY_SUMMARY_PATH = (
    CPU_ROOT /
    "study_supervision_summary.csv"
)

ENCODER_PATH = (
    SSL_ROOT /
    "ssl_training" /
    "best_encoder.pt"
)

required = {
    "manifest": MANIFEST_PATH,
    "percentiles": PERCENTILES_PATH,
    "config": CONFIG_PATH,
    "weights": WEIGHTS_PATH,
    "study_summary": STUDY_SUMMARY_PATH,
    "encoder": ENCODER_PATH,
}

print("\n" + "=" * 80)
print("REQUIRED FILES")
print("=" * 80)

for name, path in required.items():
    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(path)


# ------------------------------------------------------------
# 4. Load lightweight metadata
# ------------------------------------------------------------

manifest_df = pd.read_parquet(
    MANIFEST_PATH
)

weights_df = pd.read_csv(
    WEIGHTS_PATH
)

study_summary_df = pd.read_csv(
    STUDY_SUMMARY_PATH
)

with open(CONFIG_PATH, "r") as f:
    training_config = json.load(f)


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET CHECK")
print("=" * 80)

print("Manifest rows:", len(manifest_df))
print(
    "Unique studies:",
    manifest_df["StudyInstanceUID"].nunique()
)
print(
    "Unique series:",
    manifest_df["SeriesInstanceUID"].nunique()
)

print("\nSupervision source:")
print(
    study_summary_df[
        "SupervisionSource"
    ].value_counts()
)

print("\nTargets:")
print(training_config["targets"])

print("\nWeights:")
display(weights_df)


# ------------------------------------------------------------
# 6. Critical assertions
# ------------------------------------------------------------

assert len(manifest_df) == 770334
assert manifest_df["StudyInstanceUID"].nunique() == 4407
assert manifest_df["SeriesInstanceUID"].nunique() == 24371

assert (
    study_summary_df["SupervisionSource"]
    .value_counts()
    .get("Gold", 0)
    == 58
)

assert (
    study_summary_df["SupervisionSource"]
    .value_counts()
    .get("Pseudo", 0)
    == 4349
)

assert training_config["num_targets"] == 12
assert len(weights_df) == 12

print("\n" + "=" * 80)
print("✓ GPU ENVIRONMENT READY")
print("✓ CPU-prepared dataset loaded")
print("✓ Pretrained encoder found")
print("✓ 4,407 studies verified")
print("✓ Ready to build supervised dataloaders")
print("=" * 80)

GPU STEP 1 — ENVIRONMENT + ASSET VERIFICATION

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Competition root exists: True
CPU root exists: True
SSL root exists: True

REQUIRED FILES

manifest:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-label-production/soft_supervised_mri_cpu/combined_4407_supervised_triplets.parquet
Exists: True

percentiles:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-label-production/soft_supervised_mri_cpu/combined_24371_series_percentiles.parquet
Exists: True

config:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-label-production/soft_supervised_mri_cpu/training_config.json
Exists: True

weights:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-label-production/soft_supervised_mri_cpu/target_supervision_weights.csv
Exists: True

study_summary:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-label-production/soft_supervised_mri_cpu/study_supervision_summary.csv
Exists: Tru

,Target,GoldWeight,PseudoBaseWeight,PseudoTargetMultiplier,EffectivePseudoWeight
0,ACL,1.0,0.25,1.0,0.250
1,MCL,1.0,0.25,1.0,0.250
2,Medial Meniscus,1.0,0.25,1.0,0.250
3,Lateral Meniscus,1.0,0.25,1.0,0.250
4,Medial OA,1.0,0.25,1.0,0.250
5,Lateral OA,1.0,0.25,1.0,0.250
6,PF OA,1.0,0.25,1.0,0.250
7,Effusion,1.0,0.25,0.5,0.125
8,Synovitis,1.0,0.25,0.0,0.000
9,Baker's,1.0,0.25,1.0,0.250



✓ GPU ENVIRONMENT READY
✓ CPU-prepared dataset loaded
✓ Pretrained encoder found
✓ 4,407 studies verified
✓ Ready to build supervised dataloaders


In [2]:
# ============================================================
# GPU STEP 2 — GOLD-ONLY 5-FOLD CV SPLIT
# STUDY LEVEL — PSEUDO NEVER USED FOR VALIDATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 80)
print("GPU STEP 2 — GOLD-ONLY 5-FOLD CV")
print("=" * 80)

SEED = 123
N_FOLDS = 5

rng = np.random.default_rng(SEED)

GPU_WORK = Path(
    "/kaggle/working/soft_supervised_gpu"
)

GPU_WORK.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Target columns
# ------------------------------------------------------------

TARGETS = training_config["targets"]

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

# ------------------------------------------------------------
# 2. Get ONE row per study from manifest
# ------------------------------------------------------------

study_level = (
    manifest_df[
        [
            "StudyInstanceUID",
            "SupervisionSource",
            "PseudoSource",
        ] + target_cols
    ]
    .drop_duplicates(
        subset=["StudyInstanceUID"]
    )
    .reset_index(drop=True)
)

print("\nStudy-level rows:", len(study_level))

print("\nSource counts:")
print(
    study_level[
        "SupervisionSource"
    ].value_counts()
)

assert len(study_level) == 4407

# ------------------------------------------------------------
# 3. Separate gold and pseudo
# ------------------------------------------------------------

gold_studies = (
    study_level[
        study_level["SupervisionSource"] == "Gold"
    ]
    .copy()
    .reset_index(drop=True)
)

pseudo_studies = (
    study_level[
        study_level["SupervisionSource"] == "Pseudo"
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nGold studies:", len(gold_studies))
print("Pseudo studies:", len(pseudo_studies))

assert len(gold_studies) == 58
assert len(pseudo_studies) == 4349

# ------------------------------------------------------------
# 4. Gold label matrix
# ------------------------------------------------------------

Y = (
    gold_studies[target_cols]
    .to_numpy(dtype=np.float32)
)

# Gold targets must be true binary labels
unique_gold_values = np.unique(Y)

print("\nUnique gold target values:")
print(unique_gold_values)

assert set(unique_gold_values).issubset({0.0, 1.0})

# ------------------------------------------------------------
# 5. Show gold prevalence
# ------------------------------------------------------------

gold_positive_counts = Y.sum(axis=0)

prevalence_df = pd.DataFrame({
    "Target": TARGETS,
    "GoldPositive": gold_positive_counts.astype(int),
    "GoldNegative": (
        len(gold_studies) - gold_positive_counts
    ).astype(int),
    "Prevalence": (
        gold_positive_counts / len(gold_studies)
    ),
})

print("\n" + "=" * 80)
print("GOLD LABEL PREVALENCE")
print("=" * 80)

display(prevalence_df)

# ------------------------------------------------------------
# 6. Greedy multilabel stratification
#
# We want:
# - similar fold sizes
# - similar positives per target
#
# With only 58 gold studies, ordinary StratifiedKFold is
# unsuitable for 12 simultaneous binary labels.
# ------------------------------------------------------------

fold_sizes_target = np.array([
    len(gold_studies) // N_FOLDS
    + (1 if f < len(gold_studies) % N_FOLDS else 0)
    for f in range(N_FOLDS)
])

# Desired positive count for each target in each fold
desired_positive = (
    gold_positive_counts[None, :]
    / N_FOLDS
)

fold_positive = np.zeros(
    (N_FOLDS, len(TARGETS)),
    dtype=float
)

fold_sizes = np.zeros(
    N_FOLDS,
    dtype=int
)

fold_assignment = np.full(
    len(gold_studies),
    -1,
    dtype=int
)

# Rare-label weighting:
# studies carrying rare positive labels get placed first.
positive_frequency = np.maximum(
    gold_positive_counts,
    1
)

rarity_weights = (
    1.0 / positive_frequency
)

study_priority = (
    Y * rarity_weights
).sum(axis=1)

# Tie breaker: number of positive labels
positive_per_study = Y.sum(axis=1)

order = np.lexsort((
    rng.random(len(gold_studies)),
    -positive_per_study,
    -study_priority,
))

for idx in order:

    y = Y[idx]

    candidate_scores = []

    for fold in range(N_FOLDS):

        # Prevent fold exceeding intended size
        if fold_sizes[fold] >= fold_sizes_target[fold]:
            candidate_scores.append(np.inf)
            continue

        new_positive = (
            fold_positive[fold] + y
        )

        # Positive-balance penalty
        label_penalty = np.mean(
            (
                new_positive
                - desired_positive[0]
            ) ** 2
        )

        # Fold-size balance penalty
        new_size = fold_sizes[fold] + 1

        size_penalty = (
            new_size
            / fold_sizes_target[fold]
        ) ** 2

        score = (
            label_penalty
            + 0.10 * size_penalty
        )

        candidate_scores.append(score)

    best_score = np.min(candidate_scores)

    best_folds = np.where(
        np.isclose(
            candidate_scores,
            best_score
        )
    )[0]

    # Random tie break, deterministic via seed
    chosen_fold = rng.choice(best_folds)

    fold_assignment[idx] = chosen_fold

    fold_sizes[chosen_fold] += 1

    fold_positive[chosen_fold] += y

gold_studies["Fold"] = fold_assignment

# ------------------------------------------------------------
# 7. QC fold sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GOLD FOLD SIZES")
print("=" * 80)

print(
    gold_studies[
        "Fold"
    ].value_counts().sort_index()
)

assert (
    gold_studies["Fold"] >= 0
).all()

assert gold_studies["Fold"].nunique() == 5

# ------------------------------------------------------------
# 8. Positive counts by fold
# ------------------------------------------------------------

fold_positive_rows = []

for fold in range(N_FOLDS):

    fold_df = gold_studies[
        gold_studies["Fold"] == fold
    ]

    for target, col in zip(
        TARGETS,
        target_cols
    ):

        fold_positive_rows.append({
            "Fold": fold,
            "Target": target,
            "N": len(fold_df),
            "Positive": int(
                fold_df[col].sum()
            ),
            "Negative": int(
                len(fold_df)
                - fold_df[col].sum()
            )
        })

fold_balance_df = pd.DataFrame(
    fold_positive_rows
)

print("\n" + "=" * 80)
print("GOLD POSITIVES BY FOLD")
print("=" * 80)

positive_pivot = (
    fold_balance_df
    .pivot(
        index="Target",
        columns="Fold",
        values="Positive"
    )
)

display(positive_pivot)

# ------------------------------------------------------------
# 9. Validation feasibility check
#
# AUC cannot be computed in a fold for a target if the
# validation fold contains only one class.
# ------------------------------------------------------------

auc_feasibility = []

for fold in range(N_FOLDS):

    fold_df = gold_studies[
        gold_studies["Fold"] == fold
    ]

    for target, col in zip(
        TARGETS,
        target_cols
    ):

        n_unique = (
            fold_df[col]
            .nunique()
        )

        auc_feasibility.append({
            "Fold": fold,
            "Target": target,
            "AUC_possible": (
                n_unique == 2
            )
        })

auc_feasibility_df = pd.DataFrame(
    auc_feasibility
)

print("\nAUC-feasible target/fold pairs:")

print(
    auc_feasibility_df[
        "AUC_possible"
    ].value_counts()
)

problem_auc = auc_feasibility_df[
    ~auc_feasibility_df["AUC_possible"]
]

if len(problem_auc) > 0:

    print(
        "\n⚠ Target/fold combinations "
        "with only one validation class:"
    )

    display(problem_auc)

else:

    print(
        "\n✓ Every target has both classes "
        "in every validation fold"
    )

# ------------------------------------------------------------
# 10. Pseudo studies NEVER receive a validation fold
# ------------------------------------------------------------

pseudo_studies["Fold"] = -1

fold_assignments = pd.concat(
    [
        gold_studies[
            [
                "StudyInstanceUID",
                "SupervisionSource",
                "PseudoSource",
                "Fold",
            ]
        ],
        pseudo_studies[
            [
                "StudyInstanceUID",
                "SupervisionSource",
                "PseudoSource",
                "Fold",
            ]
        ],
    ],
    ignore_index=True
)

print("\n" + "=" * 80)
print("FULL STUDY FOLD TABLE")
print("=" * 80)

print(
    fold_assignments.groupby(
        ["SupervisionSource", "Fold"]
    ).size()
)

# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

FOLD_PATH = (
    GPU_WORK /
    "gold_5fold_assignments.csv"
)

BALANCE_PATH = (
    GPU_WORK /
    "gold_5fold_label_balance.csv"
)

fold_assignments.to_csv(
    FOLD_PATH,
    index=False
)

fold_balance_df.to_csv(
    BALANCE_PATH,
    index=False
)

print("\nSaved:")
print(FOLD_PATH)
print(BALANCE_PATH)

# ------------------------------------------------------------
# 12. Final assertions
# ------------------------------------------------------------

assert (
    fold_assignments[
        "StudyInstanceUID"
    ].nunique()
    == 4407
)

assert (
    fold_assignments.loc[
        fold_assignments[
            "SupervisionSource"
        ] == "Pseudo",
        "Fold"
    ] == -1
).all()

assert (
    fold_assignments.loc[
        fold_assignments[
            "SupervisionSource"
        ] == "Gold",
        "Fold"
    ].between(0, 4)
).all()

print("\n" + "=" * 80)
print("✓ GOLD-ONLY 5-FOLD CV CREATED")
print("✓ 58 gold studies distributed across 5 folds")
print("✓ All 4,349 pseudo studies = TRAIN ONLY")
print("✓ No pseudo study can enter validation")
print("✓ Split is study-level — no triplet leakage")
print("✓ Ready for balanced training sampling")
print("=" * 80)

GPU STEP 2 — GOLD-ONLY 5-FOLD CV

Study-level rows: 4407

Source counts:
SupervisionSource
Pseudo    4349
Gold        58
Name: count, dtype: int64

Gold studies: 58
Pseudo studies: 4349

Unique gold target values:
[0. 1.]

GOLD LABEL PREVALENCE


,Target,GoldPositive,GoldNegative,Prevalence
0,ACL,24,34,0.413793
1,MCL,9,49,0.155172
2,Medial Meniscus,26,32,0.448276
3,Lateral Meniscus,23,35,0.396552
4,Medial OA,15,43,0.258621
5,Lateral OA,11,47,0.189655
6,PF OA,21,37,0.362069
7,Effusion,35,23,0.603448
8,Synovitis,27,31,0.465517
9,Baker's,12,46,0.206897



GOLD FOLD SIZES
Fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64

GOLD POSITIVES BY FOLD


Fold,0,1,2,3,4
Target,,,,,
ACL,7,6,2,3,6
Baker's,7,4,1,0,0
Contusion,6,4,2,3,4
Effusion,11,9,2,7,6
Fracture,6,5,3,2,2
Lateral Meniscus,9,8,2,3,1
Lateral OA,6,3,0,2,0
MCL,4,1,0,2,2
Medial Meniscus,9,7,2,5,3



AUC-feasible target/fold pairs:
AUC_possible
True     53
False     7
Name: count, dtype: int64

⚠ Target/fold combinations with only one validation class:


,Fold,Target,AUC_possible
25,2,MCL,False
28,2,Medial OA,False
29,2,Lateral OA,False
45,3,Baker's,False
52,4,Medial OA,False
53,4,Lateral OA,False
57,4,Baker's,False



FULL STUDY FOLD TABLE
SupervisionSource  Fold
Gold                0        12
                    1        12
                    2        12
                    3        11
                    4        11
Pseudo             -1      4349
dtype: int64

Saved:
/kaggle/working/soft_supervised_gpu/gold_5fold_assignments.csv
/kaggle/working/soft_supervised_gpu/gold_5fold_label_balance.csv

✓ GOLD-ONLY 5-FOLD CV CREATED
✓ 58 gold studies distributed across 5 folds
✓ All 4,349 pseudo studies = TRAIN ONLY
✓ No pseudo study can enter validation
✓ Split is study-level — no triplet leakage
✓ Ready for balanced training sampling


In [3]:
# ============================================================
# GPU STEP 2B — IMPROVE GOLD 5-FOLD MULTILABEL BALANCE
# REQUIRE BOTH CLASSES FOR EVERY TARGET/FOLD IF POSSIBLE
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("GPU STEP 2B — OPTIMISING GOLD CV FOLDS")
print("=" * 80)

SEED = 123
N_FOLDS = 5
N_TRIALS = 50000

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 1. Gold labels
# ------------------------------------------------------------

gold_cv = gold_studies.copy().reset_index(drop=True)

Y = gold_cv[target_cols].to_numpy(dtype=np.int8)

n = len(gold_cv)

fold_capacities = np.array(
    [12, 12, 12, 11, 11],
    dtype=int
)

assert fold_capacities.sum() == n

total_pos = Y.sum(axis=0)
total_neg = n - total_pos

print("\nTotal positives:")
for target, pos in zip(TARGETS, total_pos):
    print(f"{target:<20} {int(pos)}")

# ------------------------------------------------------------
# 2. Score an assignment
#
# Priority 1:
#   every fold has both positive and negative gold examples
#   for every target.
#
# Priority 2:
#   balance prevalence across folds.
# ------------------------------------------------------------

def evaluate_assignment(assign):

    pos = np.zeros(
        (N_FOLDS, len(TARGETS)),
        dtype=np.int32
    )

    sizes = np.zeros(
        N_FOLDS,
        dtype=np.int32
    )

    for f in range(N_FOLDS):

        mask = (assign == f)

        sizes[f] = mask.sum()

        pos[f] = Y[mask].sum(axis=0)

    neg = sizes[:, None] - pos

    # Every target needs both classes in validation
    invalid = (
        (pos == 0) |
        (neg == 0)
    )

    n_invalid = int(invalid.sum())

    # Desired positive count by fold size
    expected_pos = (
        sizes[:, None]
        * (total_pos[None, :] / n)
    )

    # Normalise so rare and common targets matter comparably
    denom = np.maximum(
        expected_pos,
        1.0
    )

    balance_error = np.mean(
        ((pos - expected_pos) / denom) ** 2
    )

    # Huge penalty for AUC-impossible cells
    score = (
        n_invalid * 10000.0
        + balance_error
    )

    return score, n_invalid, pos, neg


# ------------------------------------------------------------
# 3. Random search
#
# 58 studies is small enough that many candidate partitions
# can be evaluated quickly.
# ------------------------------------------------------------

base_fold_vector = np.concatenate([
    np.full(cap, f, dtype=np.int8)
    for f, cap in enumerate(fold_capacities)
])

best_score = np.inf
best_invalid = None
best_assign = None
best_pos = None
best_neg = None

for trial in range(N_TRIALS):

    assign = rng.permutation(
        base_fold_vector
    )

    (
        score,
        n_invalid,
        pos,
        neg
    ) = evaluate_assignment(assign)

    if score < best_score:

        best_score = score
        best_invalid = n_invalid
        best_assign = assign.copy()
        best_pos = pos.copy()
        best_neg = neg.copy()

        # Ideal outcome achieved.
        # Keep searching a little unnecessary once balance
        # is already excellent, but zero invalid is the key.
        if (
            best_invalid == 0
            and best_score < 0.12
        ):
            break


print("\nTrials evaluated:", trial + 1)
print("Best AUC-invalid cells:", best_invalid)
print("Best score:", best_score)


# ------------------------------------------------------------
# 4. Local swap improvement
#
# Preserve fold sizes while trying pairwise study swaps.
# ------------------------------------------------------------

improved = True
passes = 0

while improved and passes < 20:

    improved = False
    passes += 1

    order = rng.permutation(n)

    for ai in range(n):

        i = order[ai]

        for aj in range(ai + 1, n):

            j = order[aj]

            if best_assign[i] == best_assign[j]:
                continue

            candidate = best_assign.copy()

            candidate[i], candidate[j] = (
                candidate[j],
                candidate[i]
            )

            (
                score,
                n_invalid,
                pos,
                neg
            ) = evaluate_assignment(candidate)

            if score < best_score - 1e-12:

                best_score = score
                best_invalid = n_invalid
                best_assign = candidate
                best_pos = pos
                best_neg = neg

                improved = True


print("Local optimisation passes:", passes)
print("Final AUC-invalid cells:", best_invalid)
print("Final score:", best_score)


# ------------------------------------------------------------
# 5. Apply improved fold assignment
# ------------------------------------------------------------

gold_cv["Fold"] = best_assign.astype(int)

print("\n" + "=" * 80)
print("FINAL GOLD FOLD SIZES")
print("=" * 80)

print(
    gold_cv["Fold"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 6. Positive counts
# ------------------------------------------------------------

positive_table = pd.DataFrame(
    best_pos.T,
    index=TARGETS,
    columns=[
        f"Fold {f}"
        for f in range(N_FOLDS)
    ]
)

print("\n" + "=" * 80)
print("FINAL POSITIVES BY FOLD")
print("=" * 80)

display(positive_table)


# ------------------------------------------------------------
# 7. Negative counts
# ------------------------------------------------------------

negative_table = pd.DataFrame(
    best_neg.T,
    index=TARGETS,
    columns=[
        f"Fold {f}"
        for f in range(N_FOLDS)
    ]
)

print("\nFINAL NEGATIVES BY FOLD:")
display(negative_table)


# ------------------------------------------------------------
# 8. AUC feasibility
# ------------------------------------------------------------

feasible = (
    (best_pos > 0) &
    (best_neg > 0)
)

print("\n" + "=" * 80)
print("AUC FEASIBILITY")
print("=" * 80)

print(
    "Feasible target/fold pairs:",
    int(feasible.sum()),
    "/",
    feasible.size
)

if feasible.all():

    print(
        "✓ ALL 60 target/fold combinations "
        "contain both classes"
    )

else:

    print("⚠ Remaining impossible cells:")

    problems = []

    for f in range(N_FOLDS):

        for t_idx, target in enumerate(TARGETS):

            if not feasible[f, t_idx]:

                problems.append({
                    "Fold": f,
                    "Target": target,
                    "Positive":
                        int(best_pos[f, t_idx]),
                    "Negative":
                        int(best_neg[f, t_idx]),
                })

    display(pd.DataFrame(problems))


# ------------------------------------------------------------
# 9. Recreate complete fold table
# ------------------------------------------------------------

gold_fold_table = gold_cv[
    [
        "StudyInstanceUID",
        "SupervisionSource",
        "PseudoSource",
    ]
].copy()

gold_fold_table["Fold"] = (
    gold_cv["Fold"].values
)

pseudo_fold_table = pseudo_studies[
    [
        "StudyInstanceUID",
        "SupervisionSource",
        "PseudoSource",
    ]
].copy()

pseudo_fold_table["Fold"] = -1

fold_assignments = pd.concat(
    [
        gold_fold_table,
        pseudo_fold_table,
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 10. Critical leakage checks
# ------------------------------------------------------------

assert (
    fold_assignments[
        "StudyInstanceUID"
    ].nunique()
    == 4407
)

assert (
    fold_assignments.loc[
        fold_assignments[
            "SupervisionSource"
        ] == "Pseudo",
        "Fold"
    ] == -1
).all()

assert (
    fold_assignments.loc[
        fold_assignments[
            "SupervisionSource"
        ] == "Gold",
        "Fold"
    ].between(0, 4)
).all()

assert (
    gold_cv[
        "StudyInstanceUID"
    ].duplicated().sum()
    == 0
)


# ------------------------------------------------------------
# 11. Save improved assignment
# ------------------------------------------------------------

FOLD_PATH = (
    GPU_WORK /
    "gold_5fold_assignments.csv"
)

fold_assignments.to_csv(
    FOLD_PATH,
    index=False
)

positive_table.to_csv(
    GPU_WORK /
    "gold_5fold_positive_balance.csv"
)

negative_table.to_csv(
    GPU_WORK /
    "gold_5fold_negative_balance.csv"
)

print("\nSaved improved folds:")
print(FOLD_PATH)

print("\n" + "=" * 80)

if feasible.all():

    print("✓ IMPROVED GOLD 5-FOLD CV PASSED")
    print("✓ ALL 60 target/fold AUCs are computable")

else:

    print(
        "⚠ Best split retained, but some "
        "target/fold AUCs remain unavailable"
    )

print("✓ Study-level split")
print("✓ 4,349 pseudo studies remain TRAIN-ONLY")
print("✓ No leakage")
print("=" * 80)

GPU STEP 2B — OPTIMISING GOLD CV FOLDS

Total positives:
ACL                  24
MCL                  9
Medial Meniscus      26
Lateral Meniscus     23
Medial OA            15
Lateral OA           11
PF OA                21
Effusion             35
Synovitis            27
Baker's              12
Contusion            19
Fracture             18

Trials evaluated: 1
Best AUC-invalid cells: 0
Best score: 0.08858426686280471
Local optimisation passes: 2
Final AUC-invalid cells: 0
Final score: 0.013907106893486414

FINAL GOLD FOLD SIZES
Fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64

FINAL POSITIVES BY FOLD


,Fold 0,Fold 1,Fold 2,Fold 3,Fold 4
ACL,5,5,5,5,4
MCL,2,2,2,1,2
Medial Meniscus,6,5,5,5,5
Lateral Meniscus,5,4,5,5,4
Medial OA,3,3,3,3,3
Lateral OA,2,3,2,2,2
PF OA,4,5,4,4,4
Effusion,7,7,8,7,6
Synovitis,6,6,6,4,5
Baker's,2,3,3,2,2



FINAL NEGATIVES BY FOLD:


,Fold 0,Fold 1,Fold 2,Fold 3,Fold 4
ACL,7,7,7,6,7
MCL,10,10,10,10,9
Medial Meniscus,6,7,7,6,6
Lateral Meniscus,7,8,7,6,7
Medial OA,9,9,9,8,8
Lateral OA,10,9,10,9,9
PF OA,8,7,8,7,7
Effusion,5,5,4,4,5
Synovitis,6,6,6,7,6
Baker's,10,9,9,9,9



AUC FEASIBILITY
Feasible target/fold pairs: 60 / 60
✓ ALL 60 target/fold combinations contain both classes

Saved improved folds:
/kaggle/working/soft_supervised_gpu/gold_5fold_assignments.csv

✓ IMPROVED GOLD 5-FOLD CV PASSED
✓ ALL 60 target/fold AUCs are computable
✓ Study-level split
✓ 4,349 pseudo studies remain TRAIN-ONLY
✓ No leakage


In [4]:
# ============================================================
# GPU STEP 3 — BALANCED TRAINING SAMPLING
# GOLD OVERSAMPLED, PSEUDO CONTROLLED
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("GPU STEP 3 — BALANCED TRAINING SAMPLING")
print("=" * 80)

SEED = 123
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 1. Attach fold assignment to manifest
# ------------------------------------------------------------

manifest_train = manifest_df.merge(
    fold_assignments[
        ["StudyInstanceUID", "Fold"]
    ],
    on="StudyInstanceUID",
    how="left",
    validate="many_to_one"
)

assert manifest_train["Fold"].isna().sum() == 0

# ------------------------------------------------------------
# 2. Sampling plan
#
# Pseudo:
#   keep broad coverage, but only a few triplets per series
#
# Gold:
#   sample much more heavily so true supervision is seen often
# ------------------------------------------------------------

PSEUDO_TRIPLETS_PER_SERIES = 4
GOLD_TRIPLETS_PER_SERIES = 16

print("\nSampling plan:")
print(
    "Pseudo triplets per series:",
    PSEUDO_TRIPLETS_PER_SERIES
)
print(
    "Gold triplets per series:",
    GOLD_TRIPLETS_PER_SERIES
)

# ------------------------------------------------------------
# 3. Helper
# ------------------------------------------------------------

def sample_per_series(df, n_per_series, seed):

    sampled_parts = []

    local_rng = np.random.default_rng(seed)

    for _, g in df.groupby(
        "SeriesInstanceUID",
        sort=False
    ):

        n = min(
            n_per_series,
            len(g)
        )

        idx = local_rng.choice(
            len(g),
            size=n,
            replace=False
        )

        sampled_parts.append(
            g.iloc[idx]
        )

    return pd.concat(
        sampled_parts,
        ignore_index=True
    )

# ------------------------------------------------------------
# 4. Build one training epoch for each fold
#
# Validation:
#   gold studies in current fold only
#
# Training:
#   all pseudo
#   + gold studies from the other 4 folds
# ------------------------------------------------------------

epoch_manifests = {}

for fold in range(5):

    print("\n" + "=" * 80)
    print(f"FOLD {fold}")
    print("=" * 80)

    train_pool = manifest_train[
        (
            manifest_train["SupervisionSource"] == "Pseudo"
        )
        |
        (
            (manifest_train["SupervisionSource"] == "Gold")
            &
            (manifest_train["Fold"] != fold)
        )
    ].copy()

    val_pool = manifest_train[
        (
            manifest_train["SupervisionSource"] == "Gold"
        )
        &
        (manifest_train["Fold"] == fold)
    ].copy()

    # --------------------------------------------------------
    # Sample pseudo
    # --------------------------------------------------------

    pseudo_pool = train_pool[
        train_pool["SupervisionSource"] == "Pseudo"
    ]

    sampled_pseudo = sample_per_series(
        pseudo_pool,
        PSEUDO_TRIPLETS_PER_SERIES,
        seed=SEED + fold
    )

    # --------------------------------------------------------
    # Sample gold
    # --------------------------------------------------------

    gold_pool = train_pool[
        train_pool["SupervisionSource"] == "Gold"
    ]

    sampled_gold = sample_per_series(
        gold_pool,
        GOLD_TRIPLETS_PER_SERIES,
        seed=SEED + 100 + fold
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    epoch_train = pd.concat(
        [
            sampled_pseudo,
            sampled_gold,
        ],
        ignore_index=True
    )

    epoch_train = epoch_train.sample(
        frac=1.0,
        random_state=SEED + fold
    ).reset_index(drop=True)

    epoch_manifests[fold] = {
        "train": epoch_train,
        "val": val_pool,
    }

    print(
        "Train rows:",
        len(epoch_train)
    )

    print(
        "  Pseudo:",
        (
            epoch_train[
                "SupervisionSource"
            ] == "Pseudo"
        ).sum()
    )

    print(
        "  Gold:",
        (
            epoch_train[
                "SupervisionSource"
            ] == "Gold"
        ).sum()
    )

    print(
        "Validation rows:",
        len(val_pool)
    )

    print(
        "Validation studies:",
        val_pool[
            "StudyInstanceUID"
        ].nunique()
    )

    print(
        "Training studies:",
        epoch_train[
            "StudyInstanceUID"
        ].nunique()
    )

# ------------------------------------------------------------
# 5. Fold-0 detailed QC
# ------------------------------------------------------------

train0 = epoch_manifests[0]["train"]
val0 = epoch_manifests[0]["val"]

print("\n" + "=" * 80)
print("FOLD 0 DETAILED QC")
print("=" * 80)

print("\nTrain source counts:")
print(
    train0[
        "SupervisionSource"
    ].value_counts()
)

print("\nTrain unique studies by source:")
print(
    train0[
        [
            "StudyInstanceUID",
            "SupervisionSource"
        ]
    ]
    .drop_duplicates()
    ["SupervisionSource"]
    .value_counts()
)

print("\nValidation source counts:")
print(
    val0[
        "SupervisionSource"
    ].value_counts()
)

# ------------------------------------------------------------
# 6. Leakage check
# ------------------------------------------------------------

for fold in range(5):

    tr = epoch_manifests[fold]["train"]
    va = epoch_manifests[fold]["val"]

    train_ids = set(
        tr["StudyInstanceUID"]
    )

    val_ids = set(
        va["StudyInstanceUID"]
    )

    overlap = train_ids.intersection(
        val_ids
    )

    assert len(overlap) == 0

    assert (
        va["SupervisionSource"] == "Gold"
    ).all()

print("\n" + "=" * 80)
print("✓ BALANCED SAMPLING READY")
print("✓ All pseudo studies contribute to training")
print("✓ Gold studies are heavily oversampled")
print("✓ Validation remains gold-only")
print("✓ No study leakage")
print("✓ Ready to build PyTorch dataset/dataloader")
print("=" * 80)

GPU STEP 3 — BALANCED TRAINING SAMPLING

Sampling plan:
Pseudo triplets per series: 4
Gold triplets per series: 16

FOLD 0
Train rows: 100525
  Pseudo: 96140
  Gold: 4385
Validation rows: 1803
Validation studies: 12
Training studies: 4395

FOLD 1
Train rows: 100305
  Pseudo: 96140
  Gold: 4165
Validation rows: 1928
Validation studies: 12
Training studies: 4395

FOLD 2
Train rows: 100292
  Pseudo: 96140
  Gold: 4152
Validation rows: 2383
Validation studies: 12
Training studies: 4395

FOLD 3
Train rows: 100351
  Pseudo: 96140
  Gold: 4211
Validation rows: 2181
Validation studies: 11
Training studies: 4396

FOLD 4
Train rows: 100451
  Pseudo: 96140
  Gold: 4311
Validation rows: 1561
Validation studies: 11
Training studies: 4396

FOLD 0 DETAILED QC

Train source counts:
SupervisionSource
Pseudo    96140
Gold       4385
Name: count, dtype: int64

Train unique studies by source:
SupervisionSource
Pseudo    4349
Gold        46
Name: count, dtype: int64

Validation source counts:
SupervisionSo

In [5]:
# ============================================================
# GPU STEP 4 — SUPERVISED 2.5D DATASET + LOADER SMOKE TEST
# NO TRAINING YET
# ============================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import pydicom
import cv2

import torch
from torch.utils.data import Dataset, DataLoader

print("=" * 80)
print("GPU STEP 4 — SUPERVISED 2.5D DATASET")
print("=" * 80)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

IMAGE_SIZE = 224
TARGET_SPACING = 0.5

TARGETS = training_config["targets"]

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

# pseudo multipliers, locked from our config
pseudo_target_multiplier = np.array(
    [
        training_config[
            "pseudo_target_multipliers"
        ][t]
        for t in TARGETS
    ],
    dtype=np.float32
)

PSEUDO_BASE_WEIGHT = float(
    training_config[
        "supervision"
    ][
        "pseudo"
    ][
        "relative_weight"
    ]
)

GOLD_WEIGHT = float(
    training_config[
        "supervision"
    ][
        "gold"
    ][
        "relative_weight"
    ]
)

print("Gold weight:", GOLD_WEIGHT)
print("Pseudo base weight:", PSEUDO_BASE_WEIGHT)

print("\nPseudo target multipliers:")
for t, w in zip(
    TARGETS,
    pseudo_target_multiplier
):
    print(f"{t:<20} {w:.3f}")


# ------------------------------------------------------------
# Helper: DICOM pixels
# ------------------------------------------------------------

def read_dicom_pixels(path):

    ds = pydicom.dcmread(
        path,
        force=True
    )

    arr = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    arr = (
        arr * slope
        + intercept
    )

    return arr


# ------------------------------------------------------------
# Helper: P1/P99 normalization
# ------------------------------------------------------------

def normalize_image(
    arr,
    p1,
    p99
):

    arr = np.clip(
        arr,
        p1,
        p99
    )

    denom = max(
        p99 - p1,
        1e-6
    )

    arr = (
        arr - p1
    ) / denom

    return arr.astype(
        np.float32
    )


# ------------------------------------------------------------
# Helper: spacing-aware resize
# ------------------------------------------------------------

def resize_to_spacing(
    arr,
    spacing_y,
    spacing_x,
    target_spacing=TARGET_SPACING
):

    h, w = arr.shape

    # Defensive fallback
    if (
        not np.isfinite(spacing_y)
        or not np.isfinite(spacing_x)
        or spacing_y <= 0
        or spacing_x <= 0
    ):
        return arr

    new_h = max(
        1,
        int(
            round(
                h
                * spacing_y
                / target_spacing
            )
        )
    )

    new_w = max(
        1,
        int(
            round(
                w
                * spacing_x
                / target_spacing
            )
        )
    )

    arr = cv2.resize(
        arr,
        (new_w, new_h),
        interpolation=cv2.INTER_LINEAR
    )

    return arr


# ------------------------------------------------------------
# Helper: centre crop / pad
# ------------------------------------------------------------

def centre_crop_pad(
    arr,
    size=IMAGE_SIZE
):

    h, w = arr.shape

    # -------------------------
    # crop height
    # -------------------------

    if h > size:

        start = (
            h - size
        ) // 2

        arr = arr[
            start:start + size,
            :
        ]

    # -------------------------
    # crop width
    # -------------------------

    h, w = arr.shape

    if w > size:

        start = (
            w - size
        ) // 2

        arr = arr[
            :,
            start:start + size
        ]

    # -------------------------
    # pad
    # -------------------------

    h, w = arr.shape

    pad_h = (
        size - h
    )

    pad_w = (
        size - w
    )

    top = (
        pad_h // 2
    )

    bottom = (
        pad_h - top
    )

    left = (
        pad_w // 2
    )

    right = (
        pad_w - left
    )

    arr = np.pad(
        arr,
        (
            (top, bottom),
            (left, right)
        ),
        mode="constant",
        constant_values=0
    )

    return arr


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

class KneeSupervisedDataset(
    Dataset
):

    def __init__(
        self,
        df,
        augment=False
    ):

        self.df = (
            df.reset_index(
                drop=True
            )
        )

        self.augment = augment

    def __len__(self):

        return len(self.df)

    def _load_slice(
        self,
        path,
        p1,
        p99,
        spacing_y,
        spacing_x
    ):

        arr = read_dicom_pixels(
            path
        )

        arr = normalize_image(
            arr,
            p1,
            p99
        )

        arr = resize_to_spacing(
            arr,
            spacing_y,
            spacing_x
        )

        arr = centre_crop_pad(
            arr,
            IMAGE_SIZE
        )

        return arr

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        p1 = float(
            row["P1"]
        )

        p99 = float(
            row["P99"]
        )

        spacing_y = float(
            row["PixelSpacingY"]
        )

        spacing_x = float(
            row["PixelSpacingX"]
        )

        # -----------------------------------
        # previous / centre / next
        # -----------------------------------

        prev_img = self._load_slice(
            row["PreviousPath"],
            p1,
            p99,
            spacing_y,
            spacing_x
        )

        centre_img = self._load_slice(
            row["CentrePath"],
            p1,
            p99,
            spacing_y,
            spacing_x
        )

        next_img = self._load_slice(
            row["NextPath"],
            p1,
            p99,
            spacing_y,
            spacing_x
        )

        image = np.stack(
            [
                prev_img,
                centre_img,
                next_img
            ],
            axis=0
        ).astype(
            np.float32
        )

        # -----------------------------------
        # lightweight MRI-safe augmentation
        # SAME spatial transform on all 3
        # channels
        # -----------------------------------

        if self.augment:

            # horizontal flip
            if random.random() < 0.5:

                image = image[
                    :,
                    :,
                    ::-1
                ].copy()

            # small intensity scaling
            if random.random() < 0.3:

                scale = random.uniform(
                    0.90,
                    1.10
                )

                image = np.clip(
                    image * scale,
                    0.0,
                    1.0
                )

        # -----------------------------------
        # Targets
        # -----------------------------------

        target = row[
            target_cols
        ].to_numpy(
            dtype=np.float32
        )

        # -----------------------------------
        # Per-target supervision weights
        # -----------------------------------

        if (
            row[
                "SupervisionSource"
            ]
            == "Gold"
        ):

            loss_weight = np.full(
                len(TARGETS),
                GOLD_WEIGHT,
                dtype=np.float32
            )

        else:

            loss_weight = (
                PSEUDO_BASE_WEIGHT
                * pseudo_target_multiplier
            ).astype(
                np.float32
            )

        return {
            "image":
                torch.from_numpy(
                    image
                ),

            "target":
                torch.from_numpy(
                    target
                ),

            "loss_weight":
                torch.from_numpy(
                    loss_weight
                ),

            "study_uid":
                row[
                    "StudyInstanceUID"
                ],

            "series_uid":
                row[
                    "SeriesInstanceUID"
                ],

            "source":
                row[
                    "SupervisionSource"
                ],

            "plane":
                row[
                    "Anatomical_Plane"
                ],
        }


# ------------------------------------------------------------
# Smoke test on Fold 0
# ------------------------------------------------------------

train0 = epoch_manifests[
    0
]["train"]

val0 = epoch_manifests[
    0
]["val"]

train_dataset = (
    KneeSupervisedDataset(
        train0,
        augment=True
    )
)

val_dataset = (
    KneeSupervisedDataset(
        val0,
        augment=False
    )
)

print("\nTrain dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))


# ------------------------------------------------------------
# Small loader first
# ------------------------------------------------------------

SMOKE_BATCH_SIZE = 8

smoke_loader = DataLoader(
    train_dataset,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("\nLoading one batch...")

batch = next(
    iter(smoke_loader)
)

# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BATCH QC")
print("=" * 80)

print(
    "Image shape:",
    batch["image"].shape
)

print(
    "Target shape:",
    batch["target"].shape
)

print(
    "Loss-weight shape:",
    batch["loss_weight"].shape
)

print(
    "\nImage dtype:",
    batch["image"].dtype
)

print(
    "Image min:",
    batch["image"].min().item()
)

print(
    "Image max:",
    batch["image"].max().item()
)

print(
    "\nTarget min:",
    batch["target"].min().item()
)

print(
    "Target max:",
    batch["target"].max().item()
)

print(
    "\nSources:",
    batch["source"]
)

print(
    "\nPlanes:",
    batch["plane"]
)

print(
    "\nExample loss weights:"
)

print(
    batch[
        "loss_weight"
    ][0]
)


# ------------------------------------------------------------
# Critical assertions
# ------------------------------------------------------------

assert (
    batch[
        "image"
    ].shape
    ==
    (
        SMOKE_BATCH_SIZE,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    )
)

assert (
    batch[
        "target"
    ].shape
    ==
    (
        SMOKE_BATCH_SIZE,
        12
    )
)

assert (
    batch[
        "loss_weight"
    ].shape
    ==
    (
        SMOKE_BATCH_SIZE,
        12
    )
)

assert torch.isfinite(
    batch["image"]
).all()

assert torch.isfinite(
    batch["target"]
).all()

assert torch.isfinite(
    batch["loss_weight"]
).all()

assert (
    batch[
        "image"
    ].min()
    >= 0
)

assert (
    batch[
        "image"
    ].max()
    <= 1.1
)

print("\n" + "=" * 80)
print("✓ SUPERVISED DATASET PASSED")
print("✓ 2.5D shape = [B, 3, 224, 224]")
print("✓ Continuous targets loaded")
print("✓ Gold/pseudo loss weights loaded")
print("✓ MRI normalization valid")
print("✓ Ready to load pretrained encoder")
print("=" * 80)

GPU STEP 4 — SUPERVISED 2.5D DATASET
Gold weight: 1.0
Pseudo base weight: 0.25

Pseudo target multipliers:
ACL                  1.000
MCL                  1.000
Medial Meniscus      1.000
Lateral Meniscus     1.000
Medial OA            1.000
Lateral OA           1.000
PF OA                1.000
Effusion             0.500
Synovitis            0.000
Baker's              1.000
Contusion            1.000
Fracture             1.000

Train dataset: 100525
Val dataset: 1803

Loading one batch...

BATCH QC
Image shape: torch.Size([8, 3, 224, 224])
Target shape: torch.Size([8, 12])
Loss-weight shape: torch.Size([8, 12])

Image dtype: torch.float32
Image min: 0.0
Image max: 1.0

Target min: 0.0
Target max: 1.0

Sources: ['Pseudo', 'Pseudo', 'Pseudo', 'Gold', 'Pseudo', 'Pseudo', 'Pseudo', 'Pseudo']

Planes: ['Axial', 'Coronal', 'Sagittal', 'Axial', 'Sagittal', 'Coronal', 'Coronal', 'Axial']

Example loss weights:
tensor([0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.1250, 0.0000,
    

In [6]:
# ============================================================
# GPU STEP 5A — INSPECT PRETRAINED ENCODER CHECKPOINT
# NO TRAINING YET
# ============================================================

import torch
from collections import OrderedDict

print("=" * 80)
print("GPU STEP 5A — INSPECT PRETRAINED ENCODER")
print("=" * 80)

print("\nEncoder path:")
print(ENCODER_PATH)

checkpoint = torch.load(
    ENCODER_PATH,
    map_location="cpu"
)

print("\nLoaded object type:")
print(type(checkpoint))

# ------------------------------------------------------------
# 1. Inspect checkpoint structure
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    print("\nTop-level keys:")

    for key in checkpoint.keys():
        print(" ", key)

else:
    print("\nCheckpoint is not a dictionary.")


# ------------------------------------------------------------
# 2. Try to locate state_dict
# ------------------------------------------------------------

state_dict = None

if isinstance(checkpoint, dict):

    possible_keys = [
        "state_dict",
        "model_state_dict",
        "encoder_state_dict",
        "encoder",
        "model",
    ]

    for key in possible_keys:

        if key in checkpoint:

            candidate = checkpoint[key]

            if isinstance(candidate, dict):

                state_dict = candidate

                print(
                    f"\nUsing state dict from key: {key}"
                )

                break

    # If checkpoint itself looks like a state dict
    if state_dict is None:

        tensor_values = [
            isinstance(v, torch.Tensor)
            for v in checkpoint.values()
        ]

        if len(tensor_values) > 0 and all(tensor_values):

            state_dict = checkpoint

            print(
                "\nCheckpoint itself appears "
                "to be a state_dict"
            )

elif isinstance(checkpoint, OrderedDict):

    state_dict = checkpoint


if state_dict is None:

    raise RuntimeError(
        "Could not identify encoder state_dict."
    )


# ------------------------------------------------------------
# 3. Show state dict keys
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STATE DICT")
print("=" * 80)

print("Number of tensors:", len(state_dict))

keys = list(state_dict.keys())

print("\nFirst 30 keys:")

for key in keys[:30]:

    print(
        f"{key:<55}",
        tuple(state_dict[key].shape)
    )


print("\nLast 20 keys:")

for key in keys[-20:]:

    print(
        f"{key:<55}",
        tuple(state_dict[key].shape)
    )


# ------------------------------------------------------------
# 4. Search for architecture clues
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ARCHITECTURE CLUES")
print("=" * 80)

clue_patterns = [
    "conv1",
    "layer1",
    "layer2",
    "layer3",
    "layer4",
    "fc",
    "classifier",
    "projection",
    "projector",
    "backbone",
    "encoder",
]

for pattern in clue_patterns:

    matching = [
        k for k in keys
        if pattern.lower() in k.lower()
    ]

    if matching:

        print(f"\n[{pattern}]")

        for key in matching[:10]:

            print(
                f"{key:<55}",
                tuple(state_dict[key].shape)
            )


# ------------------------------------------------------------
# 5. Detect common ResNet signatures
# ------------------------------------------------------------

resnet_like = any(
    "layer4" in k
    for k in keys
)

print("\nResNet-like state dict:", resnet_like)

# Conv1 gives useful architecture information
conv1_keys = [
    k for k in keys
    if k.endswith("conv1.weight")
]

print("\nConv1 candidates:")

for key in conv1_keys:

    print(
        key,
        tuple(state_dict[key].shape)
    )


# ------------------------------------------------------------
# 6. Save reference in memory
# ------------------------------------------------------------

encoder_state_dict = state_dict

print("\n" + "=" * 80)
print("✓ CHECKPOINT INSPECTION COMPLETE")
print("✓ encoder_state_dict loaded into memory")
print("✓ No model architecture guessed yet")
print("✓ No training performed")
print("=" * 80)

GPU STEP 5A — INSPECT PRETRAINED ENCODER

Encoder path:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-clean-ssl-training/ssl_training/best_encoder.pt

Loaded object type:
<class 'collections.OrderedDict'>

Top-level keys:
  conv1.weight
  bn1.weight
  bn1.bias
  bn1.running_mean
  bn1.running_var
  bn1.num_batches_tracked
  layer1.0.conv1.weight
  layer1.0.bn1.weight
  layer1.0.bn1.bias
  layer1.0.bn1.running_mean
  layer1.0.bn1.running_var
  layer1.0.bn1.num_batches_tracked
  layer1.0.conv2.weight
  layer1.0.bn2.weight
  layer1.0.bn2.bias
  layer1.0.bn2.running_mean
  layer1.0.bn2.running_var
  layer1.0.bn2.num_batches_tracked
  layer1.1.conv1.weight
  layer1.1.bn1.weight
  layer1.1.bn1.bias
  layer1.1.bn1.running_mean
  layer1.1.bn1.running_var
  layer1.1.bn1.num_batches_tracked
  layer1.1.conv2.weight
  layer1.1.bn2.weight
  layer1.1.bn2.bias
  layer1.1.bn2.running_mean
  layer1.1.bn2.running_var
  layer1.1.bn2.num_batches_tracked
  layer2.0.conv1.weight
  layer2.0.bn1.weight
 

In [7]:
# ============================================================
# GPU STEP 5B — RECONSTRUCT SSL RESNET18 + 12-TARGET HEAD
# FORWARD PASS ONLY — NO TRAINING
# ============================================================

import torch
import torch.nn as nn
import torchvision.models as models

print("=" * 80)
print("GPU STEP 5B — RECONSTRUCT PRETRAINED MRI MODEL")
print("=" * 80)

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", DEVICE)


# ------------------------------------------------------------
# 1. Reconstruct exact SSL encoder architecture
# ------------------------------------------------------------

encoder = models.resnet18(
    weights=None
)

# SSL checkpoint contains the ResNet feature extractor
# but no ImageNet classification FC layer.
encoder.fc = nn.Identity()


# ------------------------------------------------------------
# 2. Load our SSL weights
# ------------------------------------------------------------

load_result = encoder.load_state_dict(
    encoder_state_dict,
    strict=True
)

print("\nSSL encoder load result:")
print(load_result)

print(
    "\n✓ Pretrained SSL encoder loaded with strict=True"
)


# ------------------------------------------------------------
# 3. Confirm feature dimension
# ------------------------------------------------------------

encoder = encoder.to(DEVICE)
encoder.eval()

with torch.no_grad():

    dummy = torch.zeros(
        2,
        3,
        224,
        224,
        device=DEVICE
    )

    features = encoder(dummy)

print("\nEncoder feature shape:")
print(features.shape)

assert features.shape == (2, 512)


# ------------------------------------------------------------
# 4. Build 12-target MRI classifier
# ------------------------------------------------------------

class KneeMRIClassifier(nn.Module):

    def __init__(
        self,
        pretrained_state_dict,
        num_targets=12,
        dropout=0.20
    ):

        super().__init__()

        # ----------------------------------------
        # ResNet18 feature encoder
        # ----------------------------------------

        self.encoder = models.resnet18(
            weights=None
        )

        self.encoder.fc = nn.Identity()

        self.encoder.load_state_dict(
            pretrained_state_dict,
            strict=True
        )

        # ----------------------------------------
        # New supervised classification head
        # ----------------------------------------

        self.dropout = nn.Dropout(
            p=dropout
        )

        self.classifier = nn.Linear(
            512,
            num_targets
        )

    def forward(self, x):

        features = self.encoder(x)

        features = self.dropout(
            features
        )

        logits = self.classifier(
            features
        )

        return logits


# ------------------------------------------------------------
# 5. Instantiate
# ------------------------------------------------------------

model = KneeMRIClassifier(
    pretrained_state_dict=encoder_state_dict,
    num_targets=12,
    dropout=0.20
)

model = model.to(DEVICE)


# ------------------------------------------------------------
# 6. Real MRI batch forward pass
# ------------------------------------------------------------

images = batch["image"].to(
    DEVICE,
    non_blocking=True
)

targets = batch["target"].to(
    DEVICE,
    non_blocking=True
)

loss_weights = batch[
    "loss_weight"
].to(
    DEVICE,
    non_blocking=True
)

model.eval()

with torch.no_grad():

    logits = model(
        images
    )

    probabilities = torch.sigmoid(
        logits
    )


# ------------------------------------------------------------
# 7. QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REAL MRI FORWARD PASS")
print("=" * 80)

print(
    "Input:",
    images.shape
)

print(
    "Logits:",
    logits.shape
)

print(
    "Probabilities:",
    probabilities.shape
)

print(
    "\nLogit range:",
    float(logits.min()),
    "to",
    float(logits.max())
)

print(
    "Probability range:",
    float(probabilities.min()),
    "to",
    float(probabilities.max())
)

print(
    "\nTarget shape:",
    targets.shape
)

print(
    "Loss-weight shape:",
    loss_weights.shape
)


# ------------------------------------------------------------
# 8. Parameter counts
# ------------------------------------------------------------

encoder_params = sum(
    p.numel()
    for p in model.encoder.parameters()
)

head_params = sum(
    p.numel()
    for p in model.classifier.parameters()
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n" + "=" * 80)
print("MODEL SIZE")
print("=" * 80)

print(
    f"Encoder parameters: {encoder_params:,}"
)

print(
    f"New head parameters: {head_params:,}"
)

print(
    f"Total parameters:   {total_params:,}"
)


# ------------------------------------------------------------
# 9. Critical assertions
# ------------------------------------------------------------

assert logits.shape == (
    images.shape[0],
    12
)

assert probabilities.shape == (
    images.shape[0],
    12
)

assert torch.isfinite(
    logits
).all()

assert torch.isfinite(
    probabilities
).all()

assert (
    probabilities >= 0
).all()

assert (
    probabilities <= 1
).all()


print("\n" + "=" * 80)
print("✓ SSL RESNET18 RESTORED EXACTLY")
print("✓ 512-D MRI features confirmed")
print("✓ New 12-target head attached")
print("✓ Real MRI batch forward pass successful")
print("✓ No training performed yet")
print("✓ Ready to define weighted soft-label loss")
print("=" * 80)

GPU STEP 5B — RECONSTRUCT PRETRAINED MRI MODEL

Device: cuda:0

SSL encoder load result:
<All keys matched successfully>

✓ Pretrained SSL encoder loaded with strict=True

Encoder feature shape:
torch.Size([2, 512])

REAL MRI FORWARD PASS
Input: torch.Size([8, 3, 224, 224])
Logits: torch.Size([8, 12])
Probabilities: torch.Size([8, 12])

Logit range: -2.319267749786377 to 2.860942840576172
Probability range: 0.08953974395990372 to 0.9458816051483154

Target shape: torch.Size([8, 12])
Loss-weight shape: torch.Size([8, 12])

MODEL SIZE
Encoder parameters: 11,176,512
New head parameters: 6,156
Total parameters:   11,182,668

✓ SSL RESNET18 RESTORED EXACTLY
✓ 512-D MRI features confirmed
✓ New 12-target head attached
✓ Real MRI batch forward pass successful
✓ No training performed yet
✓ Ready to define weighted soft-label loss


In [8]:
# ============================================================
# GPU STEP 6 — WEIGHTED SOFT-LABEL BCE + BACKWARD SMOKE TEST
# NO FULL TRAINING YET
# ============================================================

import torch
import torch.nn as nn

print("=" * 80)
print("GPU STEP 6 — WEIGHTED SOFT-LABEL LOSS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Element-wise BCE with logits
# ------------------------------------------------------------

bce_elementwise = nn.BCEWithLogitsLoss(
    reduction="none"
)

# ------------------------------------------------------------
# 2. Forward pass on real batch
# ------------------------------------------------------------

model.train()

images = batch["image"].to(
    DEVICE,
    non_blocking=True
)

targets = batch["target"].to(
    DEVICE,
    non_blocking=True
)

loss_weights = batch[
    "loss_weight"
].to(
    DEVICE,
    non_blocking=True
)

logits = model(images)

raw_loss = bce_elementwise(
    logits,
    targets
)

print("\nRaw BCE shape:")
print(raw_loss.shape)

assert raw_loss.shape == targets.shape

# ------------------------------------------------------------
# 3. Apply per-sample / per-target supervision weights
# ------------------------------------------------------------

weighted_loss = (
    raw_loss
    * loss_weights
)

# Normalize by total active weight,
# not by total tensor elements.
active_weight_sum = (
    loss_weights.sum()
)

loss = (
    weighted_loss.sum()
    / active_weight_sum.clamp_min(1e-8)
)

print("\nWeighted loss:")
print(float(loss))

print(
    "Active supervision weight:",
    float(active_weight_sum)
)

# ------------------------------------------------------------
# 4. Check Synovitis pseudo masking
# ------------------------------------------------------------

synovitis_idx = TARGETS.index(
    "Synovitis"
)

effusion_idx = TARGETS.index(
    "Effusion"
)

print("\nExample per-row weights:")

for i in range(
    min(8, len(batch["source"]))
):

    print(
        f"{i:>2} | "
        f"{batch['source'][i]:<6} | "
        f"Effusion={loss_weights[i, effusion_idx].item():.3f} | "
        f"Synovitis={loss_weights[i, synovitis_idx].item():.3f}"
    )

# ------------------------------------------------------------
# 5. One optimizer step only
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.encoder.parameters(),
            "lr": 1e-4,
        },
        {
            "params":
                model.classifier.parameters(),
            "lr": 5e-4,
        },
    ],
    weight_decay=1e-4
)

optimizer.zero_grad(
    set_to_none=True
)

loss.backward()

# ------------------------------------------------------------
# 6. Gradient checks
# ------------------------------------------------------------

encoder_grad_norm = torch.sqrt(
    sum(
        (
            p.grad.detach().float().norm() ** 2
        )
        for p in model.encoder.parameters()
        if p.grad is not None
    )
)

head_grad_norm = torch.sqrt(
    sum(
        (
            p.grad.detach().float().norm() ** 2
        )
        for p in model.classifier.parameters()
        if p.grad is not None
    )
)

print("\nGradient norms:")
print(
    "Encoder:",
    float(encoder_grad_norm)
)
print(
    "Classifier head:",
    float(head_grad_norm)
)

assert torch.isfinite(
    encoder_grad_norm
)

assert torch.isfinite(
    head_grad_norm
)

assert encoder_grad_norm > 0
assert head_grad_norm > 0

# ------------------------------------------------------------
# 7. Optimizer smoke step
# ------------------------------------------------------------

optimizer.step()

print("\n✓ One optimizer step completed")

# ------------------------------------------------------------
# 8. Final checks
# ------------------------------------------------------------

assert torch.isfinite(loss)

assert (
    loss_weights[:, synovitis_idx]
    >= 0
).all()

assert (
    loss_weights[:, effusion_idx]
    >= 0
).all()

print("\n" + "=" * 80)
print("✓ WEIGHTED SOFT-LABEL LOSS PASSED")
print("✓ Continuous pseudo targets supported")
print("✓ Gold weight applied")
print("✓ Pseudo weight applied")
print("✓ Effusion reduced")
print("✓ Synovitis pseudo supervision masked")
print("✓ Gradients flow through encoder + classifier")
print("✓ Optimizer step successful")
print("✓ Ready for real fold training")
print("=" * 80)

GPU STEP 6 — WEIGHTED SOFT-LABEL LOSS

Raw BCE shape:
torch.Size([8, 12])

Weighted loss:
0.702333927154541
Active supervision weight: 30.375

Example per-row weights:
 0 | Pseudo | Effusion=0.125 | Synovitis=0.000
 1 | Pseudo | Effusion=0.125 | Synovitis=0.000
 2 | Pseudo | Effusion=0.125 | Synovitis=0.000
 3 | Gold   | Effusion=1.000 | Synovitis=1.000
 4 | Pseudo | Effusion=0.125 | Synovitis=0.000
 5 | Pseudo | Effusion=0.125 | Synovitis=0.000
 6 | Pseudo | Effusion=0.125 | Synovitis=0.000
 7 | Pseudo | Effusion=0.125 | Synovitis=0.000


/tmp/ipykernel_58/3472178787.py:77: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(float(loss))



Gradient norms:
Encoder: 0.3735210597515106
Classifier head: 1.6965562105178833

✓ One optimizer step completed

✓ WEIGHTED SOFT-LABEL LOSS PASSED
✓ Continuous pseudo targets supported
✓ Gold weight applied
✓ Pseudo weight applied
✓ Effusion reduced
✓ Synovitis pseudo supervision masked
✓ Gradients flow through encoder + classifier
✓ Optimizer step successful
✓ Ready for real fold training


In [9]:
# ============================================================
# GPU STEP 7 — FOLD 0 PILOT TRAINING
# 3 EPOCHS + GOLD-ONLY VALIDATION
# ============================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import roc_auc_score

print("=" * 80)
print("GPU STEP 7 — FOLD 0 PILOT TRAINING")
print("=" * 80)

FOLD = 0
EPOCHS = 3
BATCH_SIZE = 64
NUM_WORKERS = 4

FOLD_DIR = (
    GPU_WORK /
    f"fold_{FOLD}"
)

FOLD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    FOLD_DIR /
    "best_model.pt"
)

HISTORY_PATH = (
    FOLD_DIR /
    "training_history.csv"
)

# ------------------------------------------------------------
# 1. Fold-0 data
# ------------------------------------------------------------

train_df = epoch_manifests[FOLD]["train"].copy()
val_df = epoch_manifests[FOLD]["val"].copy()

print("\nTrain rows:", len(train_df))
print("Val rows:", len(val_df))
print(
    "Val studies:",
    val_df["StudyInstanceUID"].nunique()
)

# ------------------------------------------------------------
# 2. Datasets / loaders
# ------------------------------------------------------------

train_dataset = KneeSupervisedDataset(
    train_df,
    augment=True
)

val_dataset = KneeSupervisedDataset(
    val_df,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("\nTrain batches:", len(train_loader))
print("Val batches:", len(val_loader))

# ------------------------------------------------------------
# 3. Fresh model from SSL checkpoint
# ------------------------------------------------------------

model = KneeMRIClassifier(
    pretrained_state_dict=encoder_state_dict,
    num_targets=12,
    dropout=0.20
).to(DEVICE)

# ------------------------------------------------------------
# 4. Loss
# ------------------------------------------------------------

bce_elementwise = nn.BCEWithLogitsLoss(
    reduction="none"
)

# ------------------------------------------------------------
# 5. Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.encoder.parameters(),
            "lr": 1e-4,
        },
        {
            "params":
                model.classifier.parameters(),
            "lr": 5e-4,
        },
    ],
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 6. LR scheduler
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# ------------------------------------------------------------
# 7. AMP
# ------------------------------------------------------------

scaler = torch.amp.GradScaler(
    "cuda"
)

# ------------------------------------------------------------
# 8. Training helper
# ------------------------------------------------------------

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0
    n_batches = 0

    for batch in loader:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            DEVICE,
            non_blocking=True
        )

        loss_weights = batch[
            "loss_weight"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda"
        ):

            logits = model(
                images
            )

            raw_loss = (
                bce_elementwise(
                    logits,
                    targets
                )
            )

            weighted_loss = (
                raw_loss
                * loss_weights
            )

            loss = (
                weighted_loss.sum()
                /
                loss_weights.sum().clamp_min(
                    1e-8
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.detach().item()
        )

        n_batches += 1

    return (
        running_loss
        / max(n_batches, 1)
    )

# ------------------------------------------------------------
# 9. Validation helper
#
# IMPORTANT:
# Aggregate triplet predictions to STUDY LEVEL
# before computing AUC.
# ------------------------------------------------------------

@torch.no_grad()
def validate_gold(
    model,
    loader
):

    model.eval()

    rows = []

    for batch in loader:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            images
        )

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        targets = batch[
            "target"
        ].cpu().numpy()

        study_uids = batch[
            "study_uid"
        ]

        for i in range(
            len(study_uids)
        ):

            row = {
                "StudyInstanceUID":
                    study_uids[i]
            }

            for j, target in enumerate(
                TARGETS
            ):

                row[
                    f"{target}_pred"
                ] = probs[i, j]

                row[
                    f"{target}_true"
                ] = targets[i, j]

            rows.append(row)

    pred_df = pd.DataFrame(
        rows
    )

    # ----------------------------------------
    # Study-level aggregation
    # ----------------------------------------

    agg_dict = {}

    for target in TARGETS:

        agg_dict[
            f"{target}_pred"
        ] = "mean"

        agg_dict[
            f"{target}_true"
        ] = "first"

    study_pred_df = (
        pred_df
        .groupby(
            "StudyInstanceUID",
            as_index=False
        )
        .agg(
            agg_dict
        )
    )

    # ----------------------------------------
    # Per-target AUC
    # ----------------------------------------

    aucs = {}

    for target in TARGETS:

        y_true = study_pred_df[
            f"{target}_true"
        ].values

        y_pred = study_pred_df[
            f"{target}_pred"
        ].values

        auc = roc_auc_score(
            y_true,
            y_pred
        )

        aucs[target] = float(
            auc
        )

    macro_auc = float(
        np.mean(
            list(
                aucs.values()
            )
        )
    )

    return (
        macro_auc,
        aucs,
        study_pred_df
    )

# ------------------------------------------------------------
# 10. Train
# ------------------------------------------------------------

history = []

best_macro_auc = -np.inf

for epoch in range(
    1,
    EPOCHS + 1
):

    print("\n" + "=" * 80)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 80)

    start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler
    )

    (
        macro_auc,
        aucs,
        study_pred_df
    ) = validate_gold(
        model,
        val_loader
    )

    scheduler.step()

    elapsed = (
        time.time()
        - start
    )

    current_lr_encoder = (
        optimizer
        .param_groups[0]["lr"]
    )

    current_lr_head = (
        optimizer
        .param_groups[1]["lr"]
    )

    print(
        f"\nTrain loss:   {train_loss:.5f}"
    )

    print(
        f"Gold macro AUC: {macro_auc:.5f}"
    )

    print(
        f"Time: {elapsed / 60:.1f} min"
    )

    print(
        f"LR encoder: {current_lr_encoder:.7f}"
    )

    print(
        f"LR head:    {current_lr_head:.7f}"
    )

    print("\nPer-target AUC:")

    for target in TARGETS:

        print(
            f"{target:<20} "
            f"{aucs[target]:.4f}"
        )

    # ----------------------------------------
    # Save history
    # ----------------------------------------

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "macro_auc": macro_auc,
        "time_minutes":
            elapsed / 60,
        "lr_encoder":
            current_lr_encoder,
        "lr_head":
            current_lr_head,
    }

    for target in TARGETS:

        row[
            f"auc_{target}"
        ] = aucs[target]

    history.append(row)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    # ----------------------------------------
    # Best checkpoint
    # ----------------------------------------

    if macro_auc > best_macro_auc:

        best_macro_auc = (
            macro_auc
        )

        torch.save(
            {
                "fold": FOLD,
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "macro_auc":
                    macro_auc,
                "target_aucs":
                    aucs,
                "targets":
                    TARGETS,
            },
            BEST_MODEL_PATH
        )

        study_pred_df.to_csv(
            FOLD_DIR /
            "best_val_predictions.csv",
            index=False
        )

        print(
            "\n✓ New best Fold-0 checkpoint saved"
        )

    print(
        f"\nBest macro AUC so far: "
        f"{best_macro_auc:.5f}"
    )

# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("✓ FOLD 0 PILOT TRAINING COMPLETE")
print(
    f"✓ Best gold macro AUC: "
    f"{best_macro_auc:.5f}"
)
print("✓ Best model saved:")
print(BEST_MODEL_PATH)
print("✓ Training history saved:")
print(HISTORY_PATH)
print("=" * 80)

GPU STEP 7 — FOLD 0 PILOT TRAINING

Train rows: 100525
Val rows: 1803
Val studies: 12

Train batches: 1570
Val batches: 29

EPOCH 1/3

Train loss:   0.54724
Gold macro AUC: 0.67114
Time: 18.3 min
LR encoder: 0.0000750
LR head:    0.0003750

Per-target AUC:
ACL                  0.6000
MCL                  0.7500
Medial Meniscus      0.7500
Lateral Meniscus     0.5143
Medial OA            0.7407
Lateral OA           0.6500
PF OA                0.7812
Effusion             1.0000
Synovitis            0.6111
Baker's              0.5000
Contusion            0.6875
Fracture             0.4688

✓ New best Fold-0 checkpoint saved

Best macro AUC so far: 0.67114

EPOCH 2/3

Train loss:   0.52652
Gold macro AUC: 0.69730
Time: 19.2 min
LR encoder: 0.0000250
LR head:    0.0001250

Per-target AUC:
ACL                  0.7714
MCL                  0.8500
Medial Meniscus      0.6944
Lateral Meniscus     0.4857
Medial OA            0.8148
Lateral OA           0.8500
PF OA                0.6875
Effusion 

In [10]:
# ============================================================
# GPU STEP 8 — FULL 5-FOLD SOFT-SUPERVISED TRAINING
# 3 EPOCHS PER FOLD
# GOLD-ONLY OOF VALIDATION
# FRESH TRIPLET SAMPLING EACH EPOCH
# ============================================================

from pathlib import Path
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

print("=" * 80)
print("GPU STEP 8 — FULL 5-FOLD TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_FOLDS = 5
EPOCHS = 3

BATCH_SIZE = 64
NUM_WORKERS = 4

PSEUDO_TRIPLETS_PER_SERIES = 4
GOLD_TRIPLETS_PER_SERIES = 16

ENCODER_LR = 1e-4
HEAD_LR = 5e-4

WEIGHT_DECAY = 1e-4

SEED = 123

FULL_CV_DIR = (
    GPU_WORK /
    "full_5fold_cv"
)

FULL_CV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nDevice:", DEVICE)
print("Epochs per fold:", EPOCHS)
print("Batch size:", BATCH_SIZE)


# ============================================================
# 1. Reproducibility helper
# ============================================================

def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 2. Fresh epoch sampler
# ============================================================

def sample_per_series_fresh(
    df,
    n_per_series,
    seed
):

    rng = np.random.default_rng(seed)

    pieces = []

    for _, g in df.groupby(
        "SeriesInstanceUID",
        sort=False
    ):

        n = min(
            n_per_series,
            len(g)
        )

        idx = rng.choice(
            len(g),
            size=n,
            replace=False
        )

        pieces.append(
            g.iloc[idx]
        )

    return pd.concat(
        pieces,
        ignore_index=True
    )


def build_epoch_train_df(
    fold,
    epoch,
    seed=SEED
):

    # ----------------------------------------
    # Training pool:
    # all pseudo
    # + gold from other folds
    # ----------------------------------------

    pool = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Pseudo"
        )
        |
        (
            (
                manifest_train[
                    "SupervisionSource"
                ] == "Gold"
            )
            &
            (
                manifest_train[
                    "Fold"
                ] != fold
            )
        )
    ].copy()

    pseudo_pool = pool[
        pool[
            "SupervisionSource"
        ] == "Pseudo"
    ]

    gold_pool = pool[
        pool[
            "SupervisionSource"
        ] == "Gold"
    ]

    # Different seed every fold + epoch
    base_seed = (
        seed
        + fold * 1000
        + epoch * 100
    )

    sampled_pseudo = (
        sample_per_series_fresh(
            pseudo_pool,
            PSEUDO_TRIPLETS_PER_SERIES,
            base_seed
        )
    )

    sampled_gold = (
        sample_per_series_fresh(
            gold_pool,
            GOLD_TRIPLETS_PER_SERIES,
            base_seed + 50
        )
    )

    epoch_df = pd.concat(
        [
            sampled_pseudo,
            sampled_gold
        ],
        ignore_index=True
    )

    epoch_df = epoch_df.sample(
        frac=1.0,
        random_state=base_seed
    ).reset_index(drop=True)

    return epoch_df


# ============================================================
# 3. Validation helper
# ============================================================

@torch.no_grad()
def validate_gold_full(
    model,
    loader
):

    model.eval()

    rows = []

    for batch in tqdm(
        loader,
        desc="Validation",
        leave=False
    ):

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        targets = batch[
            "target"
        ].cpu().numpy()

        study_uids = batch[
            "study_uid"
        ]

        for i in range(
            len(study_uids)
        ):

            row = {
                "StudyInstanceUID":
                    study_uids[i]
            }

            for j, target in enumerate(
                TARGETS
            ):

                row[
                    f"{target}_pred"
                ] = probs[i, j]

                row[
                    f"{target}_true"
                ] = targets[i, j]

            rows.append(row)

    triplet_pred_df = pd.DataFrame(
        rows
    )

    # ----------------------------------------
    # Aggregate MRI triplets to STUDY level
    # ----------------------------------------

    agg = {}

    for target in TARGETS:

        agg[
            f"{target}_pred"
        ] = "mean"

        agg[
            f"{target}_true"
        ] = "first"

    study_pred_df = (
        triplet_pred_df
        .groupby(
            "StudyInstanceUID",
            as_index=False
        )
        .agg(agg)
    )

    aucs = {}

    for target in TARGETS:

        y_true = study_pred_df[
            f"{target}_true"
        ].values

        y_pred = study_pred_df[
            f"{target}_pred"
        ].values

        aucs[target] = float(
            roc_auc_score(
                y_true,
                y_pred
            )
        )

    macro_auc = float(
        np.mean(
            list(
                aucs.values()
            )
        )
    )

    return (
        macro_auc,
        aucs,
        study_pred_df
    )


# ============================================================
# 4. Training helper with live progress
# ============================================================

def train_epoch_full(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
    fold
):

    model.train()

    running = 0.0
    n_batches = 0

    progress = tqdm(
        loader,
        desc=(
            f"Fold {fold} "
            f"Epoch {epoch}"
        )
    )

    for batch in progress:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            DEVICE,
            non_blocking=True
        )

        loss_weights = batch[
            "loss_weight"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda"
        ):

            logits = model(
                images
            )

            raw_loss = (
                bce_elementwise(
                    logits,
                    targets
                )
            )

            weighted_loss = (
                raw_loss
                * loss_weights
            )

            loss = (
                weighted_loss.sum()
                /
                loss_weights
                .sum()
                .clamp_min(
                    1e-8
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running += (
            loss.detach().item()
        )

        n_batches += 1

        progress.set_postfix(
            loss=f"{running / n_batches:.4f}"
        )

    return (
        running
        / max(
            n_batches,
            1
        )
    )


# ============================================================
# 5. Main 5-fold loop
# ============================================================

all_fold_histories = []

best_fold_predictions = []

fold_best_scores = {}

for fold in range(
    N_FOLDS
):

    print("\n\n" + "#" * 80)
    print(
        f"STARTING FOLD {fold}"
    )
    print("#" * 80)

    seed_everything(
        SEED + fold
    )

    fold_dir = (
        FULL_CV_DIR /
        f"fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    best_model_path = (
        fold_dir /
        "best_model.pt"
    )

    history_path = (
        fold_dir /
        "training_history.csv"
    )

    # --------------------------------------------------------
    # Validation = gold only
    # --------------------------------------------------------

    val_df = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Gold"
        )
        &
        (
            manifest_train[
                "Fold"
            ] == fold
        )
    ].copy()

    val_dataset = (
        KneeSupervisedDataset(
            val_df,
            augment=False
        )
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    print(
        "\nValidation studies:",
        val_df[
            "StudyInstanceUID"
        ].nunique()
    )

    print(
        "Validation triplets:",
        len(val_df)
    )

    # --------------------------------------------------------
    # Fresh model from SSL encoder
    # --------------------------------------------------------

    model = KneeMRIClassifier(
        pretrained_state_dict=
            encoder_state_dict,
        num_targets=12,
        dropout=0.20
    ).to(DEVICE)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                    model.encoder.parameters(),
                "lr":
                    ENCODER_LR,
            },
            {
                "params":
                    model.classifier.parameters(),
                "lr":
                    HEAD_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY
    )

    # --------------------------------------------------------
    # Improved cosine scheduler
    #
    # T_max=6 means LR does NOT reach zero
    # during our 3-epoch run.
    # --------------------------------------------------------

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=6,
            eta_min=1e-6
        )
    )

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    best_macro_auc = -np.inf

    fold_history = []

    # ========================================================
    # Epoch loop
    # ========================================================

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        print(
            "\n" + "=" * 80
        )

        print(
            f"FOLD {fold} | "
            f"EPOCH {epoch}/{EPOCHS}"
        )

        print(
            "=" * 80
        )

        start = time.time()

        # ----------------------------------------------------
        # Fresh training sample THIS epoch
        # ----------------------------------------------------

        epoch_train_df = (
            build_epoch_train_df(
                fold=fold,
                epoch=epoch
            )
        )

        pseudo_n = (
            epoch_train_df[
                "SupervisionSource"
            ] == "Pseudo"
        ).sum()

        gold_n = (
            epoch_train_df[
                "SupervisionSource"
            ] == "Gold"
        ).sum()

        print(
            f"\nTrain triplets: "
            f"{len(epoch_train_df):,}"
        )

        print(
            f"Pseudo: {pseudo_n:,}"
        )

        print(
            f"Gold:   {gold_n:,}"
        )

        train_dataset = (
            KneeSupervisedDataset(
                epoch_train_df,
                augment=True
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=True,
            drop_last=True
        )

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        train_loss = (
            train_epoch_full(
                model,
                train_loader,
                optimizer,
                scaler,
                epoch,
                fold
            )
        )

        # ----------------------------------------------------
        # Gold validation
        # ----------------------------------------------------

        (
            macro_auc,
            aucs,
            study_pred_df
        ) = validate_gold_full(
            model,
            val_loader
        )

        elapsed = (
            time.time()
            - start
        )

        # Record LR BEFORE scheduler step
        encoder_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        head_lr = (
            optimizer
            .param_groups[1]["lr"]
        )

        print(
            f"\nTrain loss: "
            f"{train_loss:.5f}"
        )

        print(
            f"Gold macro AUC: "
            f"{macro_auc:.5f}"
        )

        print(
            f"Time: "
            f"{elapsed / 60:.1f} min"
        )

        print(
            f"Encoder LR: "
            f"{encoder_lr:.7f}"
        )

        print(
            f"Head LR: "
            f"{head_lr:.7f}"
        )

        print(
            "\nPer-target AUC:"
        )

        for target in TARGETS:

            print(
                f"{target:<20}"
                f"{aucs[target]:.4f}"
            )

        # ----------------------------------------------------
        # History
        # ----------------------------------------------------

        history_row = {
            "fold":
                fold,
            "epoch":
                epoch,
            "train_loss":
                train_loss,
            "macro_auc":
                macro_auc,
            "time_minutes":
                elapsed / 60,
            "encoder_lr":
                encoder_lr,
            "head_lr":
                head_lr,
        }

        for target in TARGETS:

            history_row[
                f"auc_{target}"
            ] = aucs[target]

        fold_history.append(
            history_row
        )

        all_fold_histories.append(
            history_row
        )

        pd.DataFrame(
            fold_history
        ).to_csv(
            history_path,
            index=False
        )

        # ----------------------------------------------------
        # Save best model
        # ----------------------------------------------------

        if macro_auc > best_macro_auc:

            best_macro_auc = (
                macro_auc
            )

            torch.save(
                {
                    "fold":
                        fold,

                    "epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "macro_auc":
                        macro_auc,

                    "target_aucs":
                        aucs,

                    "targets":
                        TARGETS,
                },
                best_model_path
            )

            best_pred = (
                study_pred_df
                .copy()
            )

            best_pred[
                "Fold"
            ] = fold

            best_pred[
                "BestEpoch"
            ] = epoch

            best_pred.to_csv(
                fold_dir /
                "best_val_predictions.csv",
                index=False
            )

            print(
                "\n✓ New best checkpoint"
            )

        print(
            f"\nBest Fold {fold} "
            f"AUC so far: "
            f"{best_macro_auc:.5f}"
        )

        # Scheduler after validation
        scheduler.step()

        # Free loader workers / some memory
        del train_loader
        del train_dataset

        torch.cuda.empty_cache()

    # ========================================================
    # Load BEST fold prediction
    # ========================================================

    fold_best_pred = pd.read_csv(
        fold_dir /
        "best_val_predictions.csv"
    )

    best_fold_predictions.append(
        fold_best_pred
    )

    fold_best_scores[fold] = (
        best_macro_auc
    )

    print(
        "\n" + "#" * 80
    )

    print(
        f"✓ FOLD {fold} COMPLETE"
    )

    print(
        f"✓ Best macro AUC: "
        f"{best_macro_auc:.5f}"
    )

    print(
        "#" * 80
    )

    # Save global history after each fold
    pd.DataFrame(
        all_fold_histories
    ).to_csv(
        FULL_CV_DIR /
        "all_fold_training_history.csv",
        index=False
    )

    # Release fold model
    del model
    del optimizer
    del scaler

    torch.cuda.empty_cache()


# ============================================================
# 6. Combine 58-study OOF predictions
# ============================================================

oof_df = pd.concat(
    best_fold_predictions,
    ignore_index=True
)

print("\n" + "=" * 80)
print("COMBINED GOLD OOF")
print("=" * 80)

print(
    "OOF rows:",
    len(oof_df)
)

print(
    "Unique gold studies:",
    oof_df[
        "StudyInstanceUID"
    ].nunique()
)

assert (
    oof_df[
        "StudyInstanceUID"
    ].nunique()
    == 58
)

assert len(
    oof_df
) == 58


# ============================================================
# 7. Full OOF AUC
# ============================================================

oof_aucs = {}

for target in TARGETS:

    y_true = oof_df[
        f"{target}_true"
    ].values

    y_pred = oof_df[
        f"{target}_pred"
    ].values

    oof_aucs[target] = float(
        roc_auc_score(
            y_true,
            y_pred
        )
    )

oof_macro_auc = float(
    np.mean(
        list(
            oof_aucs.values()
        )
    )
)


# ============================================================
# 8. Results
# ============================================================

print("\n" + "=" * 80)
print("FINAL 5-FOLD GOLD OOF RESULTS")
print("=" * 80)

print(
    f"\nOOF macro AUC: "
    f"{oof_macro_auc:.5f}"
)

print("\nPer-target OOF AUC:")

for target in TARGETS:

    print(
        f"{target:<20}"
        f"{oof_aucs[target]:.4f}"
    )


print("\nBest fold scores:")

for fold in range(5):

    print(
        f"Fold {fold}: "
        f"{fold_best_scores[fold]:.5f}"
    )


# ============================================================
# 9. Save OOF results
# ============================================================

OOF_PATH = (
    FULL_CV_DIR /
    "gold_58_oof_predictions.csv"
)

OOF_METRICS_PATH = (
    FULL_CV_DIR /
    "gold_58_oof_metrics.csv"
)

oof_df.to_csv(
    OOF_PATH,
    index=False
)

metrics_df = pd.DataFrame(
    [
        {
            "Target": target,
            "AUC": oof_aucs[target]
        }
        for target in TARGETS
    ]
)

metrics_df.loc[
    len(metrics_df)
] = {
    "Target": "MACRO",
    "AUC": oof_macro_auc
}

metrics_df.to_csv(
    OOF_METRICS_PATH,
    index=False
)

print("\nSaved OOF:")
print(OOF_PATH)

print("\nSaved metrics:")
print(OOF_METRICS_PATH)

print("\n" + "=" * 80)
print("✓ FULL 5-FOLD CV COMPLETE")
print("✓ 58 / 58 gold studies predicted OOF")
print("✓ All validation remained gold-only")
print("✓ 4,349 pseudo studies used for training")
print("✓ Best checkpoint saved for every fold")
print("✓ Ready to evaluate whether method is strong enough")
print("=" * 80)

GPU STEP 8 — FULL 5-FOLD TRAINING

Device: cuda:0
Epochs per fold: 3
Batch size: 64


################################################################################
STARTING FOLD 0
################################################################################

Validation studies: 12
Validation triplets: 1803

FOLD 0 | EPOCH 1/3

Train triplets: 100,525
Pseudo: 96,140
Gold:   4,385


Fold 0 Epoch 1:   0%|          | 0/1570 [00:00<?, ?it/s]

Validation:   0%|          | 0/29 [00:00<?, ?it/s]


Train loss: 0.54698
Gold macro AUC: 0.63489
Time: 19.5 min
Encoder LR: 0.0001000
Head LR: 0.0005000

Per-target AUC:
ACL                 0.4857
MCL                 0.7000
Medial Meniscus     0.6111
Lateral Meniscus    0.6857
Medial OA           0.8519
Lateral OA          0.8000
PF OA               0.7188
Effusion            0.8857
Synovitis           0.3611
Baker's             0.3000
Contusion           0.7188
Fracture            0.5000

✓ New best checkpoint

Best Fold 0 AUC so far: 0.63489

FOLD 0 | EPOCH 2/3

Train triplets: 100,525
Pseudo: 96,140
Gold:   4,385


Fold 0 Epoch 2:   0%|          | 0/1570 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/25 [00:00<?, ?it/s]


Train loss: 0.52809
Gold macro AUC: 0.75519
Time: 24.1 min
Encoder LR: 0.0000934
Head LR: 0.0004666

Per-target AUC:
ACL                 0.8214
MCL                 0.6111
Medial Meniscus     0.9000
Lateral Meniscus    0.8214
Medial OA           1.0000
Lateral OA          0.7222
PF OA               0.5714
Effusion            0.9333
Synovitis           0.6000
Baker's             0.6111
Contusion           0.6786
Fracture            0.7917

✓ New best checkpoint

Best Fold 4 AUC so far: 0.75519

FOLD 4 | EPOCH 3/3

Train triplets: 100,451
Pseudo: 96,140
Gold:   4,311


Fold 4 Epoch 3:   0%|          | 0/1569 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]


Train loss: 0.51539
Gold macro AUC: 0.79378
Time: 21.7 min
Encoder LR: 0.0000752
Head LR: 0.0003753

Per-target AUC:
ACL                 0.9643
MCL                 0.8333
Medial Meniscus     0.9667
Lateral Meniscus    0.6786
Medial OA           1.0000
Lateral OA          0.7778
PF OA               0.5714
Effusion            0.9333
Synovitis           0.6333
Baker's             0.5000
Contusion           0.7500
Fracture            0.9167

✓ New best checkpoint

Best Fold 4 AUC so far: 0.79378

################################################################################
✓ FOLD 4 COMPLETE
✓ Best macro AUC: 0.79378
################################################################################

COMBINED GOLD OOF
OOF rows: 58
Unique gold studies: 58

FINAL 5-FOLD GOLD OOF RESULTS

OOF macro AUC: 0.73403

Per-target OOF AUC:
ACL                 0.6667
MCL                 0.6395
Medial Meniscus     0.7488
Lateral Meniscus    0.6472
Medial OA           0.8775
Lateral OA          0.7563
P

In [11]:
# ============================================================
# GPU STEP 9 — OOF DIAGNOSTICS
# LIGHTWEIGHT / CPU
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("GPU STEP 9 — OOF DIAGNOSTICS")
print("=" * 80)

OOF_PATH = (
    FULL_CV_DIR /
    "gold_58_oof_predictions.csv"
)

METRICS_PATH = (
    FULL_CV_DIR /
    "gold_58_oof_metrics.csv"
)

oof_df = pd.read_csv(
    OOF_PATH
)

metrics_df = pd.read_csv(
    METRICS_PATH
)

print("\nOOF rows:", len(oof_df))
print(
    "Unique studies:",
    oof_df["StudyInstanceUID"].nunique()
)

# ------------------------------------------------------------
# 1. Target ranking
# ------------------------------------------------------------

target_metrics = (
    metrics_df[
        metrics_df["Target"] != "MACRO"
    ]
    .sort_values(
        "AUC",
        ascending=True
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TARGETS RANKED FROM WEAKEST TO STRONGEST")
print("=" * 80)

display(target_metrics)

# ------------------------------------------------------------
# 2. Fold-level best epochs + scores
# ------------------------------------------------------------

history = pd.read_csv(
    FULL_CV_DIR /
    "all_fold_training_history.csv"
)

best_rows = (
    history
    .sort_values(
        ["fold", "macro_auc"],
        ascending=[True, False]
    )
    .groupby(
        "fold",
        as_index=False
    )
    .first()
)

print("\n" + "=" * 80)
print("BEST EPOCH PER FOLD")
print("=" * 80)

display(
    best_rows[
        [
            "fold",
            "epoch",
            "macro_auc",
            "train_loss",
            "time_minutes",
        ]
    ]
)

# ------------------------------------------------------------
# 3. Per-target fold variation
# ------------------------------------------------------------

variation_rows = []

for target in TARGETS:

    vals = []

    col = f"auc_{target}"

    for _, row in best_rows.iterrows():
        vals.append(
            float(row[col])
        )

    variation_rows.append({
        "Target": target,
        "MeanFoldAUC": np.mean(vals),
        "StdFoldAUC": np.std(vals),
        "MinFoldAUC": np.min(vals),
        "MaxFoldAUC": np.max(vals),
        "OOF_AUC": float(
            target_metrics.loc[
                target_metrics["Target"] == target,
                "AUC"
            ].iloc[0]
        ),
    })

variation_df = pd.DataFrame(
    variation_rows
).sort_values(
    "OOF_AUC"
)

print("\n" + "=" * 80)
print("TARGET STABILITY ACROSS FOLDS")
print("=" * 80)

display(variation_df)

# ------------------------------------------------------------
# 4. Prediction separation
# ------------------------------------------------------------

separation_rows = []

for target in TARGETS:

    y = oof_df[
        f"{target}_true"
    ].values

    p = oof_df[
        f"{target}_pred"
    ].values

    pos = p[y == 1]
    neg = p[y == 0]

    separation_rows.append({
        "Target": target,
        "PositiveN": len(pos),
        "NegativeN": len(neg),
        "MeanPredPositive": np.mean(pos),
        "MeanPredNegative": np.mean(neg),
        "MedianPredPositive": np.median(pos),
        "MedianPredNegative": np.median(neg),
        "MeanSeparation":
            np.mean(pos) - np.mean(neg),
    })

separation_df = pd.DataFrame(
    separation_rows
).sort_values(
    "MeanSeparation"
)

print("\n" + "=" * 80)
print("OOF PREDICTION SEPARATION")
print("=" * 80)

display(separation_df)

# ------------------------------------------------------------
# 5. Identify priority targets
# ------------------------------------------------------------

priority_targets = (
    target_metrics[
        target_metrics["AUC"] < 0.70
    ]["Target"]
    .tolist()
)

print("\n" + "=" * 80)
print("PRIORITY TARGETS")
print("=" * 80)

print(priority_targets)

print("\n" + "=" * 80)
print("✓ OOF diagnostics complete")
print("✓ No retraining performed")
print("✓ Safe to continue before Quick Save")
print("=" * 80)

GPU STEP 9 — OOF DIAGNOSTICS

OOF rows: 58
Unique studies: 58

TARGETS RANKED FROM WEAKEST TO STRONGEST


,Target,AUC
0,Synovitis,0.627240
1,MCL,0.639456
2,Lateral Meniscus,0.647205
3,ACL,0.666667
4,PF OA,0.702703
5,Fracture,0.740278
6,Medial Meniscus,0.748798
7,Lateral OA,0.756286
8,Baker's,0.760870
9,Contusion,0.770580



BEST EPOCH PER FOLD


,fold,epoch,macro_auc,train_loss,time_minutes
0,0,3,0.715418,0.513861,20.143974
1,1,3,0.771916,0.516790,20.893064
2,2,2,0.776265,0.526665,18.217562
3,3,3,0.771032,0.518061,19.864892
4,4,3,0.793783,0.515391,21.656841



TARGET STABILITY ACROSS FOLDS


,Target,MeanFoldAUC,StdFoldAUC,MinFoldAUC,MaxFoldAUC,OOF_AUC
8,Synovitis,0.689365,0.179793,0.361111,0.833333,0.627240
1,MCL,0.716667,0.223109,0.400000,0.950000,0.639456
3,Lateral Meniscus,0.708155,0.090819,0.566667,0.828571,0.647205
0,ACL,0.715714,0.158462,0.500000,0.964286,0.666667
6,PF OA,0.689286,0.120539,0.531250,0.843750,0.702703
11,Fracture,0.783333,0.152297,0.500000,0.916667,0.740278
2,Medial Meniscus,0.770476,0.162587,0.542857,0.966667,0.748798
5,Lateral OA,0.804815,0.090051,0.700000,0.962963,0.756286
9,Baker's,0.751111,0.175344,0.500000,1.000000,0.760870
10,Contusion,0.783333,0.048591,0.750000,0.875000,0.770580



OOF PREDICTION SEPARATION


,Target,PositiveN,NegativeN,MeanPredPositive,MeanPredNegative,MedianPredPositive,MedianPredNegative,MeanSeparation
1,MCL,9,49,0.186573,0.170256,0.177964,0.163785,0.016317
3,Lateral Meniscus,23,35,0.245869,0.211732,0.243783,0.218687,0.034136
0,ACL,24,34,0.305034,0.255616,0.293779,0.257123,0.049417
11,Fracture,18,40,0.201102,0.143550,0.207503,0.124909,0.057552
9,Baker's,12,46,0.316005,0.255043,0.312502,0.260021,0.060962
2,Medial Meniscus,26,32,0.476976,0.407348,0.494578,0.383083,0.069629
10,Contusion,19,39,0.315260,0.237597,0.317146,0.234190,0.077663
8,Synovitis,27,31,0.506719,0.426009,0.491887,0.425512,0.080710
6,PF OA,21,37,0.452290,0.369094,0.444555,0.349136,0.083196
5,Lateral OA,11,47,0.320886,0.228553,0.309851,0.223807,0.092333



PRIORITY TARGETS
['Synovitis', 'MCL', 'Lateral Meniscus', 'ACL']

✓ OOF diagnostics complete
✓ No retraining performed
✓ Safe to continue before Quick Save


In [12]:
# ============================================================
# GPU STEP 10 — GOLD-ONLY REFINEMENT OF 5 BEST MODELS
# SHORT / LIGHTWEIGHT
#
# Start from Step-8 best checkpoints.
# Train ONLY on non-validation gold studies.
# Never overwrite the existing baseline models.
# ============================================================

from pathlib import Path
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print("=" * 80)
print("GPU STEP 10 — GOLD-ONLY REFINEMENT")
print("=" * 80)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

N_FOLDS = 5
REFINE_EPOCHS = 2

BATCH_SIZE = 32
NUM_WORKERS = 4

GOLD_TRIPLETS_PER_SERIES = 16

ENCODER_LR = 1e-5
HEAD_LR = 5e-5

WEIGHT_DECAY = 1e-4

SEED = 456

REFINE_DIR = (
    GPU_WORK /
    "gold_only_refinement"
)

REFINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nRefinement epochs:", REFINE_EPOCHS)
print("Encoder LR:", ENCODER_LR)
print("Head LR:", HEAD_LR)


# ============================================================
# 1. Freeze BatchNorm running statistics
# ============================================================

def freeze_batchnorm_stats(model):

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d
        ):

            module.eval()


# ============================================================
# 2. Gold-only epoch sampler
# ============================================================

def build_gold_refine_df(
    fold,
    epoch
):

    gold_pool = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Gold"
        )
        &
        (
            manifest_train[
                "Fold"
            ] != fold
        )
    ].copy()

    rng = np.random.default_rng(
        SEED
        + fold * 100
        + epoch
    )

    pieces = []

    for _, g in gold_pool.groupby(
        "SeriesInstanceUID",
        sort=False
    ):

        n = min(
            GOLD_TRIPLETS_PER_SERIES,
            len(g)
        )

        idx = rng.choice(
            len(g),
            size=n,
            replace=False
        )

        pieces.append(
            g.iloc[idx]
        )

    result = pd.concat(
        pieces,
        ignore_index=True
    )

    result = result.sample(
        frac=1,
        random_state=
            SEED + fold * 100 + epoch
    ).reset_index(drop=True)

    return result


# ============================================================
# 3. Gold-only training epoch
# ============================================================

def train_gold_epoch(
    model,
    loader,
    optimizer,
    scaler,
    fold,
    epoch
):

    model.train()

    # Important for tiny gold dataset
    freeze_batchnorm_stats(
        model.encoder
    )

    running_loss = 0.0
    batches = 0

    progress = tqdm(
        loader,
        desc=(
            f"Refine F{fold} E{epoch}"
        )
    )

    for batch in progress:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda"
        ):

            logits = model(
                images
            )

            # GOLD ONLY:
            # every target receives equal full weight
            raw_loss = (
                bce_elementwise(
                    logits,
                    targets
                )
            )

            loss = raw_loss.mean()

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        batches += 1

        running_loss += (
            loss.detach().item()
        )

        progress.set_postfix(
            loss=f"{running_loss/batches:.4f}"
        )

    return (
        running_loss /
        max(batches, 1)
    )


# ============================================================
# 4. Refine each fold
# ============================================================

refinement_results = []

refined_oof_parts = []

for fold in range(N_FOLDS):

    print("\n\n" + "#" * 80)
    print(f"REFINING FOLD {fold}")
    print("#" * 80)

    torch.manual_seed(
        SEED + fold
    )

    np.random.seed(
        SEED + fold
    )

    random.seed(
        SEED + fold
    )

    # --------------------------------------------------------
    # Existing Step-8 best checkpoint
    # --------------------------------------------------------

    baseline_path = (
        FULL_CV_DIR /
        f"fold_{fold}" /
        "best_model.pt"
    )

    checkpoint = torch.load(
        baseline_path,
        map_location="cpu"
    )

    baseline_auc = float(
        checkpoint["macro_auc"]
    )

    print(
        f"\nBaseline fold AUC: "
        f"{baseline_auc:.5f}"
    )

    # --------------------------------------------------------
    # Rebuild model
    # --------------------------------------------------------

    model = KneeMRIClassifier(
        pretrained_state_dict=
            encoder_state_dict,
        num_targets=12,
        dropout=0.20
    ).to(DEVICE)

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True
    )

    print("✓ Baseline checkpoint loaded")

    # --------------------------------------------------------
    # Validation remains SAME held-out fold
    # --------------------------------------------------------

    val_df = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Gold"
        )
        &
        (
            manifest_train[
                "Fold"
            ] == fold
        )
    ].copy()

    val_dataset = (
        KneeSupervisedDataset(
            val_df,
            augment=False
        )
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    # --------------------------------------------------------
    # Optimizer — very small LR
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                    model.encoder.parameters(),
                "lr":
                    ENCODER_LR,
            },
            {
                "params":
                    model.classifier.parameters(),
                "lr":
                    HEAD_LR,
            },
        ],
        weight_decay=
            WEIGHT_DECAY
    )

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    fold_dir = (
        REFINE_DIR /
        f"fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    refined_path = (
        fold_dir /
        "best_refined_model.pt"
    )

    best_auc = baseline_auc
    best_epoch = 0
    improved = False

    # Baseline predictions are our fallback
    best_pred_df = pd.read_csv(
        FULL_CV_DIR /
        f"fold_{fold}" /
        "best_val_predictions.csv"
    )

    # ========================================================
    # Refinement epochs
    # ========================================================

    for epoch in range(
        1,
        REFINE_EPOCHS + 1
    ):

        print(
            "\n" + "=" * 80
        )

        print(
            f"FOLD {fold} "
            f"GOLD REFINE "
            f"{epoch}/{REFINE_EPOCHS}"
        )

        print(
            "=" * 80
        )

        start = time.time()

        gold_train_df = (
            build_gold_refine_df(
                fold,
                epoch
            )
        )

        print(
            "\nGold train triplets:",
            len(gold_train_df)
        )

        print(
            "Gold train studies:",
            gold_train_df[
                "StudyInstanceUID"
            ].nunique()
        )

        train_dataset = (
            KneeSupervisedDataset(
                gold_train_df,
                augment=True
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=True,
            drop_last=True
        )

        train_loss = train_gold_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            fold,
            epoch
        )

        (
            macro_auc,
            aucs,
            study_pred_df
        ) = validate_gold_full(
            model,
            val_loader
        )

        elapsed = (
            time.time()
            - start
        )

        print(
            f"\nGold train loss: "
            f"{train_loss:.5f}"
        )

        print(
            f"Validation macro AUC: "
            f"{macro_auc:.5f}"
        )

        print(
            f"Current baseline:      "
            f"{baseline_auc:.5f}"
        )

        print(
            f"Time: "
            f"{elapsed/60:.1f} min"
        )

        print("\nPer-target AUC:")

        for target in TARGETS:

            print(
                f"{target:<20}"
                f"{aucs[target]:.4f}"
            )

        # ----------------------------------------------------
        # Only accept genuine improvement
        # ----------------------------------------------------

        if macro_auc > best_auc:

            best_auc = macro_auc
            best_epoch = epoch
            improved = True

            torch.save(
                {
                    "fold":
                        fold,

                    "refine_epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "macro_auc":
                        macro_auc,

                    "baseline_auc":
                        baseline_auc,

                    "target_aucs":
                        aucs,

                    "targets":
                        TARGETS,
                },
                refined_path
            )

            best_pred_df = (
                study_pred_df.copy()
            )

            best_pred_df[
                "Fold"
            ] = fold

            best_pred_df[
                "RefineEpoch"
            ] = epoch

            print(
                "\n✓ Refinement improved fold"
            )

        else:

            print(
                "\nNo improvement — "
                "baseline remains safer"
            )

        del train_loader
        del train_dataset

        torch.cuda.empty_cache()

    # ========================================================
    # Decide final model for this fold
    # ========================================================

    if improved:

        selected_source = (
            "GoldRefined"
        )

    else:

        selected_source = (
            "OriginalSoftSupervised"
        )

    best_pred_df[
        "SelectedModel"
    ] = selected_source

    refined_oof_parts.append(
        best_pred_df
    )

    refinement_results.append({
        "Fold":
            fold,

        "BaselineAUC":
            baseline_auc,

        "BestRefinedAUC":
            best_auc,

        "Improvement":
            best_auc - baseline_auc,

        "SelectedModel":
            selected_source,

        "BestRefineEpoch":
            best_epoch,
    })

    print("\n" + "#" * 80)

    print(
        f"Fold {fold}: "
        f"{baseline_auc:.5f} "
        f"→ {best_auc:.5f}"
    )

    print(
        "Selected:",
        selected_source
    )

    print("#" * 80)

    del model
    del optimizer
    del scaler

    torch.cuda.empty_cache()


# ============================================================
# 5. Refinement summary
# ============================================================

refinement_df = pd.DataFrame(
    refinement_results
)

print("\n" + "=" * 80)
print("REFINEMENT SUMMARY")
print("=" * 80)

display(refinement_df)


# ============================================================
# 6. Combined refined-or-baseline OOF
# ============================================================

refined_oof_df = pd.concat(
    refined_oof_parts,
    ignore_index=True
)

assert (
    refined_oof_df[
        "StudyInstanceUID"
    ].nunique()
    == 58
)

refined_aucs = {}

for target in TARGETS:

    refined_aucs[target] = (
        roc_auc_score(
            refined_oof_df[
                f"{target}_true"
            ],
            refined_oof_df[
                f"{target}_pred"
            ]
        )
    )

refined_macro_auc = float(
    np.mean(
        list(
            refined_aucs.values()
        )
    )
)

print("\n" + "=" * 80)
print("FINAL REFINED OOF RESULT")
print("=" * 80)

print(
    f"\nOriginal OOF macro AUC: "
    f"{oof_macro_auc:.5f}"
)

print(
    f"Refined OOF macro AUC:  "
    f"{refined_macro_auc:.5f}"
)

print(
    f"Change: "
    f"{refined_macro_auc-oof_macro_auc:+.5f}"
)

print("\nPer-target refined AUC:")

for target in TARGETS:

    print(
        f"{target:<20}"
        f"{refined_aucs[target]:.4f}"
    )


# ============================================================
# 7. Save results
# ============================================================

refinement_df.to_csv(
    REFINE_DIR /
    "refinement_summary.csv",
    index=False
)

refined_oof_df.to_csv(
    REFINE_DIR /
    "gold_58_refined_oof_predictions.csv",
    index=False
)

print("\n" + "=" * 80)
print("✓ GOLD-ONLY REFINEMENT COMPLETE")
print("✓ Original checkpoints preserved")
print("✓ Refined model used only if fold AUC improved")
print("✓ No pseudo labels changed")
print("✓ No API calls")
print("=" * 80)

GPU STEP 10 — GOLD-ONLY REFINEMENT

Refinement epochs: 2
Encoder LR: 1e-05
Head LR: 5e-05


################################################################################
REFINING FOLD 0
################################################################################

Baseline fold AUC: 0.71542
✓ Baseline checkpoint loaded

FOLD 0 GOLD REFINE 1/2

Gold train triplets: 4385
Gold train studies: 46


Refine F0 E1:   0%|          | 0/137 [00:00<?, ?it/s]

Validation:   0%|          | 0/57 [00:00<?, ?it/s]


Gold train loss: 0.45662
Validation macro AUC: 0.68646
Current baseline:      0.71542
Time: 1.0 min

Per-target AUC:
ACL                 0.8286
MCL                 0.7500
Medial Meniscus     0.6389
Lateral Meniscus    0.6857
Medial OA           0.8148
Lateral OA          0.7500
PF OA               0.7500
Effusion            0.8286
Synovitis           0.4722
Baker's             0.5000
Contusion           0.7812
Fracture            0.4375

No improvement — baseline remains safer

FOLD 0 GOLD REFINE 2/2

Gold train triplets: 4385
Gold train studies: 46


Refine F0 E2:   0%|          | 0/137 [00:00<?, ?it/s]

...

Refine F4 E2:   0%|          | 0/134 [00:00<?, ?it/s]

Validation:   0%|          | 0/49 [00:00<?, ?it/s]


Gold train loss: 0.44319
Validation macro AUC: 0.74521
Current baseline:      0.79378
Time: 0.9 min

Per-target AUC:
ACL                 0.9286
MCL                 0.6111
Medial Meniscus     0.9333
Lateral Meniscus    0.6786
Medial OA           1.0000
Lateral OA          0.6111
PF OA               0.5357
Effusion            0.9667
Synovitis           0.6000
Baker's             0.5000
Contusion           0.7857
Fracture            0.7917

No improvement — baseline remains safer

################################################################################
Fold 4: 0.79378 → 0.79378
Selected: OriginalSoftSupervised
################################################################################

REFINEMENT SUMMARY


,Fold,BaselineAUC,BestRefinedAUC,Improvement,SelectedModel,BestRefineEpoch
0,0,0.715418,0.715418,0.000000,OriginalSoftSupervised,0
1,1,0.771916,0.777940,0.006024,GoldRefined,1
2,2,0.776265,0.776265,0.000000,OriginalSoftSupervised,0
3,3,0.771032,0.771032,0.000000,OriginalSoftSupervised,0
4,4,0.793783,0.793783,0.000000,OriginalSoftSupervised,0



FINAL REFINED OOF RESULT

Original OOF macro AUC: 0.73403
Refined OOF macro AUC:  0.71770
Change: -0.01633

Per-target refined AUC:
ACL                 0.6507
MCL                 0.6553
Medial Meniscus     0.7500
Lateral Meniscus    0.5876
Medial OA           0.8372
Lateral OA          0.7389
PF OA               0.6718
Effusion            0.8820
Synovitis           0.6368
Baker's             0.7409
Contusion           0.7598
Fracture            0.7014

✓ GOLD-ONLY REFINEMENT COMPLETE
✓ Original checkpoints preserved
✓ Refined model used only if fold AUC improved
✓ No pseudo labels changed
✓ No API calls


In [13]:
# ============================================================
# GPU STEP 11 — OOF AGGREGATION COMPARISON
#
# NO RETRAINING
# Uses ORIGINAL Step-8 soft-supervised checkpoints only.
#
# Compare:
# 1. Triplet mean      (current baseline)
# 2. Equal-series mean
# 3. Equal-plane mean
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

print("=" * 80)
print("GPU STEP 11 — OOF AGGREGATION COMPARISON")
print("=" * 80)

AGG_DIR = (
    GPU_WORK /
    "aggregation_experiment"
)

AGG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_SIZE = 64
NUM_WORKERS = 4


# ============================================================
# 1. Inference helper
# ============================================================

@torch.no_grad()
def collect_triplet_predictions(
    model,
    loader,
    fold
):

    model.eval()

    rows = []

    for batch in tqdm(
        loader,
        desc=f"Fold {fold} inference",
        leave=False
    ):

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        targets = batch[
            "target"
        ].cpu().numpy()

        study_uids = batch[
            "study_uid"
        ]

        series_uids = batch[
            "series_uid"
        ]

        planes = batch[
            "plane"
        ]

        for i in range(
            len(study_uids)
        ):

            row = {
                "StudyInstanceUID":
                    study_uids[i],

                "SeriesInstanceUID":
                    series_uids[i],

                "Anatomical_Plane":
                    planes[i],

                "Fold":
                    fold,
            }

            for j, target in enumerate(
                TARGETS
            ):

                row[
                    f"{target}_pred"
                ] = float(
                    probs[i, j]
                )

                row[
                    f"{target}_true"
                ] = float(
                    targets[i, j]
                )

            rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 2. Run original best checkpoints across 5 validation folds
# ============================================================

all_triplet_predictions = []

for fold in range(5):

    print("\n" + "#" * 80)
    print(f"ORIGINAL MODEL — FOLD {fold}")
    print("#" * 80)

    # --------------------------------------------------------
    # IMPORTANT:
    # Original Step-8 model, NOT gold-refined model
    # --------------------------------------------------------

    checkpoint_path = (
        FULL_CV_DIR /
        f"fold_{fold}" /
        "best_model.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    print(
        "Checkpoint AUC:",
        checkpoint["macro_auc"]
    )

    model = KneeMRIClassifier(
        pretrained_state_dict=
            encoder_state_dict,
        num_targets=12,
        dropout=0.20
    ).to(DEVICE)

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True
    )

    # --------------------------------------------------------
    # Gold validation fold only
    # --------------------------------------------------------

    val_df = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Gold"
        )
        &
        (
            manifest_train[
                "Fold"
            ] == fold
        )
    ].copy()

    val_dataset = KneeSupervisedDataset(
        val_df,
        augment=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    fold_triplet_pred = (
        collect_triplet_predictions(
            model,
            val_loader,
            fold
        )
    )

    print(
        "Triplets predicted:",
        len(fold_triplet_pred)
    )

    print(
        "Studies:",
        fold_triplet_pred[
            "StudyInstanceUID"
        ].nunique()
    )

    all_triplet_predictions.append(
        fold_triplet_pred
    )

    del model
    del val_loader
    del val_dataset

    torch.cuda.empty_cache()


triplet_pred_df = pd.concat(
    all_triplet_predictions,
    ignore_index=True
)

print("\n" + "=" * 80)
print("COMPLETE TRIPLET OOF PREDICTIONS")
print("=" * 80)

print(
    "Triplets:",
    len(triplet_pred_df)
)

print(
    "Studies:",
    triplet_pred_df[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Series:",
    triplet_pred_df[
        "SeriesInstanceUID"
    ].nunique()
)

assert (
    triplet_pred_df[
        "StudyInstanceUID"
    ].nunique()
    == 58
)


# ============================================================
# 3. Helper for true-label aggregation
# ============================================================

pred_cols = [
    f"{t}_pred"
    for t in TARGETS
]

true_cols = [
    f"{t}_true"
    for t in TARGETS
]


# ============================================================
# 4. METHOD A — TRIPLET MEAN
#
# This should approximately reproduce our current 0.73403.
# ============================================================

agg_dict = {}

for col in pred_cols:
    agg_dict[col] = "mean"

for col in true_cols:
    agg_dict[col] = "first"

agg_dict["Fold"] = "first"

triplet_mean_df = (
    triplet_pred_df
    .groupby(
        "StudyInstanceUID",
        as_index=False
    )
    .agg(agg_dict)
)

triplet_mean_df[
    "Aggregation"
] = "TripletMean"


# ============================================================
# 5. METHOD B — EQUAL SERIES MEAN
#
# Triplets
#    ↓ mean
# Series
#    ↓ equal mean
# Study
#
# Every MRI series gets equal final influence.
# ============================================================

series_agg = {}

for col in pred_cols:
    series_agg[col] = "mean"

for col in true_cols:
    series_agg[col] = "first"

series_agg[
    "Anatomical_Plane"
] = "first"

series_agg["Fold"] = "first"

series_level_df = (
    triplet_pred_df
    .groupby(
        [
            "StudyInstanceUID",
            "SeriesInstanceUID"
        ],
        as_index=False
    )
    .agg(series_agg)
)

study_series_agg = {}

for col in pred_cols:
    study_series_agg[col] = "mean"

for col in true_cols:
    study_series_agg[col] = "first"

study_series_agg[
    "Fold"
] = "first"

series_mean_df = (
    series_level_df
    .groupby(
        "StudyInstanceUID",
        as_index=False
    )
    .agg(study_series_agg)
)

series_mean_df[
    "Aggregation"
] = "EqualSeriesMean"


# ============================================================
# 6. METHOD C — EQUAL PLANE MEAN
#
# Triplets
#    ↓ mean
# Series
#    ↓ mean within plane
# Plane
#    ↓ equal mean across Axial / Coronal / Sagittal
# Study
#
# Thus each anatomical plane gets equal final influence.
# ============================================================

plane_agg = {}

for col in pred_cols:
    plane_agg[col] = "mean"

for col in true_cols:
    plane_agg[col] = "first"

plane_agg[
    "Fold"
] = "first"

plane_level_df = (
    series_level_df
    .groupby(
        [
            "StudyInstanceUID",
            "Anatomical_Plane"
        ],
        as_index=False
    )
    .agg(plane_agg)
)

study_plane_agg = {}

for col in pred_cols:
    study_plane_agg[col] = "mean"

for col in true_cols:
    study_plane_agg[col] = "first"

study_plane_agg[
    "Fold"
] = "first"

plane_mean_df = (
    plane_level_df
    .groupby(
        "StudyInstanceUID",
        as_index=False
    )
    .agg(study_plane_agg)
)

plane_mean_df[
    "Aggregation"
] = "EqualPlaneMean"


# ============================================================
# 7. Scoring helper
# ============================================================

def score_oof(
    df,
    method_name
):

    aucs = {}

    for target in TARGETS:

        aucs[target] = float(
            roc_auc_score(
                df[
                    f"{target}_true"
                ],
                df[
                    f"{target}_pred"
                ]
            )
        )

    macro_auc = float(
        np.mean(
            list(
                aucs.values()
            )
        )
    )

    rows = []

    for target in TARGETS:

        rows.append({
            "Aggregation":
                method_name,

            "Target":
                target,

            "AUC":
                aucs[target],
        })

    rows.append({
        "Aggregation":
            method_name,

        "Target":
            "MACRO",

        "AUC":
            macro_auc,
    })

    return (
        macro_auc,
        aucs,
        pd.DataFrame(rows)
    )


# ============================================================
# 8. Score all three
# ============================================================

(
    triplet_macro,
    triplet_aucs,
    triplet_metrics
) = score_oof(
    triplet_mean_df,
    "TripletMean"
)

(
    series_macro,
    series_aucs,
    series_metrics
) = score_oof(
    series_mean_df,
    "EqualSeriesMean"
)

(
    plane_macro,
    plane_aucs,
    plane_metrics
) = score_oof(
    plane_mean_df,
    "EqualPlaneMean"
)


# ============================================================
# 9. Compare results
# ============================================================

comparison_df = pd.DataFrame({
    "Aggregation": [
        "TripletMean",
        "EqualSeriesMean",
        "EqualPlaneMean",
    ],

    "MacroAUC": [
        triplet_macro,
        series_macro,
        plane_macro,
    ]
})

comparison_df[
    "ChangeVsTriplet"
] = (
    comparison_df[
        "MacroAUC"
    ]
    - triplet_macro
)

comparison_df = (
    comparison_df
    .sort_values(
        "MacroAUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("AGGREGATION MACRO AUC COMPARISON")
print("=" * 80)

display(comparison_df)


# ============================================================
# 10. Per-target comparison
# ============================================================

target_compare_rows = []

for target in TARGETS:

    target_compare_rows.append({
        "Target":
            target,

        "TripletMean":
            triplet_aucs[target],

        "EqualSeriesMean":
            series_aucs[target],

        "EqualPlaneMean":
            plane_aucs[target],
    })

target_comparison_df = pd.DataFrame(
    target_compare_rows
)

target_comparison_df[
    "BestMethod"
] = (
    target_comparison_df[
        [
            "TripletMean",
            "EqualSeriesMean",
            "EqualPlaneMean"
        ]
    ]
    .idxmax(axis=1)
)

target_comparison_df[
    "BestAUC"
] = (
    target_comparison_df[
        [
            "TripletMean",
            "EqualSeriesMean",
            "EqualPlaneMean"
        ]
    ]
    .max(axis=1)
)

print("\n" + "=" * 80)
print("PER-TARGET AGGREGATION COMPARISON")
print("=" * 80)

display(
    target_comparison_df
    .sort_values(
        "BestAUC"
    )
)


# ============================================================
# 11. Save everything
# ============================================================

triplet_pred_df.to_parquet(
    AGG_DIR /
    "gold_oof_triplet_predictions.parquet",
    index=False
)

series_level_df.to_parquet(
    AGG_DIR /
    "gold_oof_series_predictions.parquet",
    index=False
)

plane_level_df.to_parquet(
    AGG_DIR /
    "gold_oof_plane_predictions.parquet",
    index=False
)

triplet_mean_df.to_csv(
    AGG_DIR /
    "oof_triplet_mean.csv",
    index=False
)

series_mean_df.to_csv(
    AGG_DIR /
    "oof_equal_series_mean.csv",
    index=False
)

plane_mean_df.to_csv(
    AGG_DIR /
    "oof_equal_plane_mean.csv",
    index=False
)

comparison_df.to_csv(
    AGG_DIR /
    "aggregation_macro_comparison.csv",
    index=False
)

target_comparison_df.to_csv(
    AGG_DIR /
    "aggregation_target_comparison.csv",
    index=False
)


# ============================================================
# 12. Final status
# ============================================================

best_method = comparison_df.iloc[0]

print("\n" + "=" * 80)
print("AGGREGATION EXPERIMENT COMPLETE")
print("=" * 80)

print(
    "\nCurrent triplet-mean AUC:",
    f"{triplet_macro:.5f}"
)

print(
    "Best aggregation:",
    best_method["Aggregation"]
)

print(
    "Best macro AUC:",
    f"{best_method['MacroAUC']:.5f}"
)

print(
    "Change:",
    f"{best_method['ChangeVsTriplet']:+.5f}"
)

print("\n✓ No retraining performed")
print("✓ Original five models untouched")
print("✓ Same 58 gold OOF studies")
print("✓ Ready to decide final aggregation strategy")
print("=" * 80)

GPU STEP 11 — OOF AGGREGATION COMPARISON

################################################################################
ORIGINAL MODEL — FOLD 0
################################################################################
Checkpoint AUC: 0.7154183201058201


Fold 0 inference:   0%|          | 0/29 [00:00<?, ?it/s]

Triplets predicted: 1803
Studies: 12

################################################################################
ORIGINAL MODEL — FOLD 1
################################################################################
Checkpoint AUC: 0.7719163359788359


Fold 1 inference:   0%|          | 0/31 [00:00<?, ?it/s]

Triplets predicted: 1928
Studies: 12

################################################################################
ORIGINAL MODEL — FOLD 2
################################################################################
Checkpoint AUC: 0.776264880952381


...

Triplets predicted: 1561
Studies: 11

COMPLETE TRIPLET OOF PREDICTIONS
Triplets: 9856
Studies: 58
Series: 336

AGGREGATION MACRO AUC COMPARISON


,Aggregation,MacroAUC,ChangeVsTriplet
0,TripletMean,0.734034,0.000000
1,EqualPlaneMean,0.732668,-0.001366
2,EqualSeriesMean,0.729258,-0.004776



PER-TARGET AGGREGATION COMPARISON


,Target,TripletMean,EqualSeriesMean,EqualPlaneMean,BestMethod,BestAUC
8,Synovitis,0.627240,0.629630,0.634409,EqualPlaneMean,0.634409
1,MCL,0.639456,0.623583,0.630385,TripletMean,0.639456
3,Lateral Meniscus,0.647205,0.647205,0.652174,EqualPlaneMean,0.652174
0,ACL,0.666667,0.640931,0.650735,TripletMean,0.666667
6,PF OA,0.702703,0.693694,0.685972,TripletMean,0.702703
2,Medial Meniscus,0.748798,0.748798,0.740385,EqualSeriesMean,0.748798
11,Fracture,0.740278,0.759722,0.751389,EqualSeriesMean,0.759722
5,Lateral OA,0.756286,0.750484,0.760155,EqualPlaneMean,0.760155
9,Baker's,0.760870,0.755435,0.748188,TripletMean,0.760870
10,Contusion,0.770580,0.740891,0.763833,TripletMean,0.770580



AGGREGATION EXPERIMENT COMPLETE

Current triplet-mean AUC: 0.73403
Best aggregation: TripletMean
Best macro AUC: 0.73403
Change: +0.00000

✓ No retraining performed
✓ Original five models untouched
✓ Same 58 gold OOF studies
✓ Ready to decide final aggregation strategy


In [1]:
# ============================================================
# RESTART 1 — LOCATE SAVED GPU RESULTS
# ============================================================

from pathlib import Path

NOTEBOOK_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37"
)

print("=" * 80)
print("SEARCHING FOR YESTERDAY'S GPU RESULTS")
print("=" * 80)

matches = list(
    NOTEBOOK_ROOT.glob(
        "*/soft_supervised_gpu/full_5fold_cv/"
        "gold_58_oof_predictions.csv"
    )
)

print("\nOOF matches:")

for p in matches:
    print("✓", p)

if len(matches) == 0:
    raise FileNotFoundError(
        "Yesterday's saved GPU notebook output is not attached."
    )

OOF_PATH = matches[0]

FULL_CV_DIR = OOF_PATH.parent

# soft_supervised_gpu/
SAVED_GPU_ROOT = FULL_CV_DIR.parent

print("\nSelected FULL_CV_DIR:")
print(FULL_CV_DIR)

print("\nExpected fold checkpoints:")

for fold in range(5):

    p = (
        FULL_CV_DIR /
        f"fold_{fold}" /
        "best_model.pt"
    )

    print(
        f"Fold {fold}:",
        p.exists()
    )

print("\n✓ Saved GPU run located")

SEARCHING FOR YESTERDAY'S GPU RESULTS

OOF matches:
✓ /kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-supervised-gpu-training/soft_supervised_gpu/full_5fold_cv/gold_58_oof_predictions.csv

Selected FULL_CV_DIR:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-supervised-gpu-training/soft_supervised_gpu/full_5fold_cv

Expected fold checkpoints:
Fold 0: True
Fold 1: True
Fold 2: True
Fold 3: True
Fold 4: True

✓ Saved GPU run located


In [2]:
# ============================================================
# RESTART 2 — RESTORE PROJECT STATE
# NO TRAINING
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as models

print("=" * 80)
print("RESTORING SOFT-SUPERVISED MRI PROJECT")
print("=" * 80)

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", DEVICE)

# ------------------------------------------------------------
# Existing CPU-prepared assets
# ------------------------------------------------------------

CPU_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-soft-label-production/"
    "soft_supervised_mri_cpu"
)

SSL_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-clean-ssl-training"
)

MANIFEST_PATH = (
    CPU_ROOT /
    "combined_4407_supervised_triplets.parquet"
)

CONFIG_PATH = (
    CPU_ROOT /
    "training_config.json"
)

ENCODER_PATH = (
    SSL_ROOT /
    "ssl_training" /
    "best_encoder.pt"
)

print("\nManifest:", MANIFEST_PATH.exists())
print("Config:", CONFIG_PATH.exists())
print("Encoder:", ENCODER_PATH.exists())

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

manifest_df = pd.read_parquet(
    MANIFEST_PATH
)

with open(CONFIG_PATH, "r") as f:
    training_config = json.load(f)

encoder_state_dict = torch.load(
    ENCODER_PATH,
    map_location="cpu"
)

TARGETS = training_config["targets"]

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

# ------------------------------------------------------------
# Reload yesterday's fold assignments
# ------------------------------------------------------------

FOLD_PATH = (
    SAVED_GPU_ROOT /
    "gold_5fold_assignments.csv"
)

print("\nFold assignments:")
print(FOLD_PATH)
print("Exists:", FOLD_PATH.exists())

fold_assignments = pd.read_csv(
    FOLD_PATH
)

# ------------------------------------------------------------
# Attach folds to full triplet manifest
# ------------------------------------------------------------

manifest_train = manifest_df.merge(
    fold_assignments[
        [
            "StudyInstanceUID",
            "Fold"
        ]
    ],
    on="StudyInstanceUID",
    how="left",
    validate="many_to_one"
)

assert (
    manifest_train["Fold"]
    .isna()
    .sum()
    == 0
)

assert (
    manifest_train[
        "StudyInstanceUID"
    ].nunique()
    == 4407
)

# ------------------------------------------------------------
# New working folder for today's experiment
# ------------------------------------------------------------

GPU_WORK = Path(
    "/kaggle/working/soft_supervised_gpu"
)

GPU_WORK.mkdir(
    parents=True,
    exist_ok=True
)

print("\n" + "=" * 80)
print("RESTORE CHECK")
print("=" * 80)

print(
    "Manifest rows:",
    len(manifest_train)
)

print(
    "Studies:",
    manifest_train[
        "StudyInstanceUID"
    ].nunique()
)

print("\nStudy folds:")

print(
    fold_assignments.groupby(
        ["SupervisionSource", "Fold"]
    ).size()
)

print("\nTargets:", TARGETS)

print("\n✓ Project state restored")
print("✓ Original 5-fold checkpoints preserved")
print("✓ No training rerun")

RESTORING SOFT-SUPERVISED MRI PROJECT

Device: cuda:0

Manifest: True
Config: True
Encoder: True

Fold assignments:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-supervised-gpu-training/soft_supervised_gpu/gold_5fold_assignments.csv
Exists: True

RESTORE CHECK
Manifest rows: 770334
Studies: 4407

Study folds:
SupervisionSource  Fold
Gold                0        12
                    1        12
                    2        12
                    3        11
                    4        11
Pseudo             -1      4349
dtype: int64

Targets: ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

✓ Project state restored
✓ Original 5-fold checkpoints preserved
✓ No training rerun


In [3]:
# ============================================================
# RESTART 3 — RESTORE DATASET + SAMPLING HELPERS
# ============================================================

import cv2
import pydicom

from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 224
TARGET_SPACING = 0.5

PSEUDO_TRIPLETS_PER_SERIES = 4
GOLD_TRIPLETS_PER_SERIES = 16

PSEUDO_BASE_WEIGHT = float(
    training_config[
        "supervision"
    ]["pseudo"]["relative_weight"]
)

GOLD_WEIGHT = float(
    training_config[
        "supervision"
    ]["gold"]["relative_weight"]
)

pseudo_target_multiplier = np.array(
    [
        training_config[
            "pseudo_target_multipliers"
        ][t]
        for t in TARGETS
    ],
    dtype=np.float32
)


# ------------------------------------------------------------
# DICOM preprocessing
# ------------------------------------------------------------

def read_dicom_pixels(path):

    ds = pydicom.dcmread(
        path,
        force=True
    )

    arr = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    return (
        arr * slope
        + intercept
    )


def normalize_image(
    arr,
    p1,
    p99
):

    arr = np.clip(
        arr,
        p1,
        p99
    )

    arr = (
        arr - p1
    ) / max(
        p99 - p1,
        1e-6
    )

    return arr.astype(
        np.float32
    )


def resize_to_spacing(
    arr,
    spacing_y,
    spacing_x,
    target_spacing=TARGET_SPACING
):

    h, w = arr.shape

    if (
        not np.isfinite(spacing_y)
        or not np.isfinite(spacing_x)
        or spacing_y <= 0
        or spacing_x <= 0
    ):
        return arr

    new_h = max(
        1,
        round(
            h * spacing_y
            / target_spacing
        )
    )

    new_w = max(
        1,
        round(
            w * spacing_x
            / target_spacing
        )
    )

    return cv2.resize(
        arr,
        (
            int(new_w),
            int(new_h)
        ),
        interpolation=cv2.INTER_LINEAR
    )


def centre_crop_pad(
    arr,
    size=IMAGE_SIZE
):

    h, w = arr.shape

    if h > size:
        s = (h - size) // 2
        arr = arr[
            s:s + size,
            :
        ]

    h, w = arr.shape

    if w > size:
        s = (w - size) // 2
        arr = arr[
            :,
            s:s + size
        ]

    h, w = arr.shape

    ph = size - h
    pw = size - w

    top = ph // 2
    bottom = ph - top

    left = pw // 2
    right = pw - left

    return np.pad(
        arr,
        (
            (top, bottom),
            (left, right)
        ),
        mode="constant"
    )


class KneeSupervisedDataset(
    Dataset
):

    def __init__(
        self,
        df,
        augment=False
    ):

        self.df = (
            df
            .reset_index(drop=True)
        )

        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load(
        self,
        path,
        p1,
        p99,
        sy,
        sx
    ):

        arr = read_dicom_pixels(
            path
        )

        arr = normalize_image(
            arr,
            p1,
            p99
        )

        arr = resize_to_spacing(
            arr,
            sy,
            sx
        )

        arr = centre_crop_pad(
            arr
        )

        return arr

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        p1 = float(row["P1"])
        p99 = float(row["P99"])

        sy = float(
            row["PixelSpacingY"]
        )

        sx = float(
            row["PixelSpacingX"]
        )

        image = np.stack(
            [
                self._load(
                    row["PreviousPath"],
                    p1, p99, sy, sx
                ),
                self._load(
                    row["CentrePath"],
                    p1, p99, sy, sx
                ),
                self._load(
                    row["NextPath"],
                    p1, p99, sy, sx
                ),
            ],
            axis=0
        ).astype(
            np.float32
        )

        if self.augment:

            if random.random() < 0.5:
                image = image[
                    :, :, ::-1
                ].copy()

            if random.random() < 0.3:

                scale = random.uniform(
                    0.90,
                    1.10
                )

                image = np.clip(
                    image * scale,
                    0,
                    1
                )

        target = row[
            target_cols
        ].to_numpy(
            dtype=np.float32
        )

        if (
            row["SupervisionSource"]
            == "Gold"
        ):

            loss_weight = np.full(
                12,
                GOLD_WEIGHT,
                dtype=np.float32
            )

        else:

            loss_weight = (
                PSEUDO_BASE_WEIGHT
                * pseudo_target_multiplier
            ).astype(
                np.float32
            )

        return {
            "image":
                torch.from_numpy(
                    image
                ),

            "target":
                torch.from_numpy(
                    target
                ),

            "loss_weight":
                torch.from_numpy(
                    loss_weight
                ),

            "study_uid":
                row["StudyInstanceUID"],

            "series_uid":
                row["SeriesInstanceUID"],

            "source":
                row["SupervisionSource"],

            "plane":
                row["Anatomical_Plane"],
        }


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

bce_elementwise = (
    nn.BCEWithLogitsLoss(
        reduction="none"
    )
)


# ------------------------------------------------------------
# Fresh per-epoch triplet sampling
# ------------------------------------------------------------

def sample_per_series_fresh(
    df,
    n_per_series,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    parts = []

    for _, g in df.groupby(
        "SeriesInstanceUID",
        sort=False
    ):

        n = min(
            n_per_series,
            len(g)
        )

        idx = rng.choice(
            len(g),
            size=n,
            replace=False
        )

        parts.append(
            g.iloc[idx]
        )

    return pd.concat(
        parts,
        ignore_index=True
    )


def build_epoch_train_df(
    fold,
    epoch,
    seed=123
):

    pool = manifest_train[
        (
            manifest_train[
                "SupervisionSource"
            ] == "Pseudo"
        )
        |
        (
            (
                manifest_train[
                    "SupervisionSource"
                ] == "Gold"
            )
            &
            (
                manifest_train[
                    "Fold"
                ] != fold
            )
        )
    ].copy()

    pseudo_pool = pool[
        pool[
            "SupervisionSource"
        ] == "Pseudo"
    ]

    gold_pool = pool[
        pool[
            "SupervisionSource"
        ] == "Gold"
    ]

    base_seed = (
        seed
        + fold * 1000
        + epoch * 100
    )

    pseudo = (
        sample_per_series_fresh(
            pseudo_pool,
            PSEUDO_TRIPLETS_PER_SERIES,
            base_seed
        )
    )

    gold = (
        sample_per_series_fresh(
            gold_pool,
            GOLD_TRIPLETS_PER_SERIES,
            base_seed + 50
        )
    )

    result = pd.concat(
        [pseudo, gold],
        ignore_index=True
    )

    return result.sample(
        frac=1,
        random_state=base_seed
    ).reset_index(drop=True)


print("=" * 80)
print("✓ Dataset restored")
print("✓ DICOM preprocessing restored")
print("✓ Weighted soft-label loss restored")
print("✓ Epoch resampling restored")
print("✓ READY FOR GPU STEP 12A")
print("=" * 80)

✓ Dataset restored
✓ DICOM preprocessing restored
✓ Weighted soft-label loss restored
✓ Epoch resampling restored
✓ READY FOR GPU STEP 12A


In [5]:
# ============================================================
# GPU STEP 13 — FINAL FULL-DATA TRAINING
#
# ALL 4,407 STUDIES
# 58 GOLD + 4,349 SOFT PSEUDO
#
# Validated original recipe:
# - SSL ResNet18
# - continuous soft pseudo labels
# - pseudo base weight = 0.25
# - Effusion pseudo multiplier = 0.50
# - Synovitis pseudo multiplier = 0.00
# - 4 pseudo triplets / series / epoch
# - 16 gold triplets / series / epoch
# - 3 epochs
#
# NO experimental branches.
# ============================================================

from pathlib import Path
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as models

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print("=" * 80)
print("GPU STEP 13 — FINAL FULL-DATA TRAINING")
print("=" * 80)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

EPOCHS = 3

BATCH_SIZE = 64
NUM_WORKERS = 4

PSEUDO_TRIPLETS_PER_SERIES = 4
GOLD_TRIPLETS_PER_SERIES = 16

ENCODER_LR = 1e-4
HEAD_LR = 5e-4

WEIGHT_DECAY = 1e-4

SEED = 2026

FINAL_DIR = Path(
    "/kaggle/working/soft_supervised_gpu/"
    "final_full_data_model"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HISTORY_PATH = (
    FINAL_DIR /
    "final_training_history.csv"
)

print("\nDevice:", DEVICE)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)


# ============================================================
# 1. Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. Final model architecture
#
# SAME model as our successful 5-fold CV.
# ============================================================

class KneeMRIClassifier(nn.Module):

    def __init__(
        self,
        pretrained_state_dict,
        num_targets=12,
        dropout=0.20
    ):

        super().__init__()

        self.encoder = models.resnet18(
            weights=None
        )

        self.encoder.fc = nn.Identity()

        self.encoder.load_state_dict(
            pretrained_state_dict,
            strict=True
        )

        self.dropout = nn.Dropout(
            p=dropout
        )

        self.classifier = nn.Linear(
            512,
            num_targets
        )

    def forward(self, x):

        features = self.encoder(x)

        features = self.dropout(
            features
        )

        logits = self.classifier(
            features
        )

        return logits


# ============================================================
# 3. Build fresh full-data epoch sample
# ============================================================

def build_final_epoch_df(
    epoch,
    seed=SEED
):

    pseudo_pool = manifest_train[
        manifest_train[
            "SupervisionSource"
        ] == "Pseudo"
    ].copy()

    gold_pool = manifest_train[
        manifest_train[
            "SupervisionSource"
        ] == "Gold"
    ].copy()

    base_seed = (
        seed
        + epoch * 100
    )

    sampled_pseudo = (
        sample_per_series_fresh(
            pseudo_pool,
            PSEUDO_TRIPLETS_PER_SERIES,
            base_seed
        )
    )

    sampled_gold = (
        sample_per_series_fresh(
            gold_pool,
            GOLD_TRIPLETS_PER_SERIES,
            base_seed + 50
        )
    )

    epoch_df = pd.concat(
        [
            sampled_pseudo,
            sampled_gold
        ],
        ignore_index=True
    )

    epoch_df = epoch_df.sample(
        frac=1.0,
        random_state=base_seed
    ).reset_index(drop=True)

    return epoch_df


# ============================================================
# 4. Training helper
# ============================================================

def train_final_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch
):

    model.train()

    running_loss = 0.0
    n_batches = 0

    progress = tqdm(
        loader,
        desc=f"Final Epoch {epoch}"
    )

    for batch in progress:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "target"
        ].to(
            DEVICE,
            non_blocking=True
        )

        loss_weights = batch[
            "loss_weight"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda"
        ):

            logits = model(
                images
            )

            raw_loss = (
                bce_elementwise(
                    logits,
                    targets
                )
            )

            weighted_loss = (
                raw_loss
                * loss_weights
            )

            loss = (
                weighted_loss.sum()
                /
                loss_weights
                .sum()
                .clamp_min(1e-8)
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.detach().item()
        )

        n_batches += 1

        progress.set_postfix(
            loss=f"{running_loss/n_batches:.4f}"
        )

    return (
        running_loss /
        max(n_batches, 1)
    )


# ============================================================
# 5. Create FRESH model from SSL encoder
#
# Important:
# We do NOT start from one particular CV fold model.
# The final model sees all 58 gold studies.
# ============================================================

final_model = KneeMRIClassifier(
    pretrained_state_dict=
        encoder_state_dict,
    num_targets=12,
    dropout=0.20
).to(DEVICE)

print("\n✓ Fresh final model created from SSL encoder")


# ============================================================
# 6. Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                final_model.encoder.parameters(),

            "lr":
                ENCODER_LR,
        },

        {
            "params":
                final_model.classifier.parameters(),

            "lr":
                HEAD_LR,
        },
    ],
    weight_decay=
        WEIGHT_DECAY
)


# ============================================================
# 7. Scheduler
#
# Same improved schedule as full CV.
# Does NOT hit zero by epoch 3.
# ============================================================

scheduler = (
    torch.optim.lr_scheduler
    .CosineAnnealingLR(
        optimizer,
        T_max=6,
        eta_min=1e-6
    )
)

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 8. Final training
# ============================================================

history = []

for epoch in range(
    1,
    EPOCHS + 1
):

    print("\n" + "=" * 80)
    print(
        f"FINAL MODEL — EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 80)

    start = time.time()

    # --------------------------------------------------------
    # Fresh MRI locations every epoch
    # --------------------------------------------------------

    epoch_train_df = (
        build_final_epoch_df(
            epoch=epoch
        )
    )

    pseudo_n = (
        epoch_train_df[
            "SupervisionSource"
        ] == "Pseudo"
    ).sum()

    gold_n = (
        epoch_train_df[
            "SupervisionSource"
        ] == "Gold"
    ).sum()

    print(
        f"\nTraining triplets: "
        f"{len(epoch_train_df):,}"
    )

    print(
        f"Pseudo triplets:   "
        f"{pseudo_n:,}"
    )

    print(
        f"Gold triplets:     "
        f"{gold_n:,}"
    )

    print(
        "\nUnique studies:"
    )

    print(
        epoch_train_df[
            "StudyInstanceUID"
        ].nunique()
    )

    # Must contain ALL 4,407 studies
    assert (
        epoch_train_df[
            "StudyInstanceUID"
        ].nunique()
        == 4407
    )

    # --------------------------------------------------------
    # Dataset / loader
    # --------------------------------------------------------

    train_dataset = (
        KneeSupervisedDataset(
            epoch_train_df,
            augment=True
        )
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True
    )

    print(
        "Training batches:",
        len(train_loader)
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train_loss = train_final_epoch(
        final_model,
        train_loader,
        optimizer,
        scaler,
        epoch
    )

    elapsed = (
        time.time()
        - start
    )

    encoder_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    head_lr = (
        optimizer
        .param_groups[1]["lr"]
    )

    print(
        f"\nTrain loss: "
        f"{train_loss:.5f}"
    )

    print(
        f"Time: "
        f"{elapsed/60:.1f} min"
    )

    print(
        f"Encoder LR: "
        f"{encoder_lr:.7f}"
    )

    print(
        f"Head LR: "
        f"{head_lr:.7f}"
    )

    # --------------------------------------------------------
    # Save EVERY epoch
    #
    # We have no validation set now, so don't pretend that
    # lowest training loss determines the best generalization.
    # --------------------------------------------------------

    checkpoint_path = (
        FINAL_DIR /
        f"final_model_epoch_{epoch}.pt"
    )

    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                final_model.state_dict(),

            "train_loss":
                train_loss,

            "targets":
                TARGETS,

            "architecture":
                "ResNet18",

            "aggregation":
                "TripletMean",

            "pseudo_weight":
                PSEUDO_BASE_WEIGHT,

            "gold_weight":
                GOLD_WEIGHT,

            "pseudo_target_multipliers":
                training_config[
                    "pseudo_target_multipliers"
                ],
        },
        checkpoint_path
    )

    print(
        "\n✓ Checkpoint saved:"
    )

    print(checkpoint_path)

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history.append({
        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "time_minutes":
            elapsed / 60,

        "encoder_lr":
            encoder_lr,

        "head_lr":
            head_lr,

        "training_triplets":
            len(epoch_train_df),

        "pseudo_triplets":
            int(pseudo_n),

        "gold_triplets":
            int(gold_n),
    })

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    # Scheduler after epoch
    scheduler.step()

    # Clean workers / RAM
    del train_loader
    del train_dataset
    del epoch_train_df

    torch.cuda.empty_cache()


# ============================================================
# 9. Default final checkpoint
#
# CV evidence:
# 4/5 folds preferred Epoch 3.
# Therefore Epoch 3 is our default final model.
# ============================================================

FINAL_MODEL_PATH = (
    FINAL_DIR /
    "final_model_epoch_3.pt"
)

assert FINAL_MODEL_PATH.exists()


# ============================================================
# 10. Final summary
# ============================================================

print("\n" + "=" * 80)
print("FINAL FULL-DATA TRAINING COMPLETE")
print("=" * 80)

print("\nTraining history:")
display(
    pd.DataFrame(history)
)

print(
    "\nDefault final checkpoint:"
)

print(
    FINAL_MODEL_PATH
)

print("\n✓ All 4,407 studies used")
print("✓ All 58 gold studies used")
print("✓ All 4,349 pseudo studies used")
print("✓ Continuous soft targets preserved")
print("✓ Original validated weighting preserved")
print("✓ TripletMean strategy preserved")
print("✓ Epoch 1/2/3 checkpoints all preserved")
print("✓ NEXT = TEST MRI PREPARATION + INFERENCE")
print("=" * 80)

GPU STEP 13 — FINAL FULL-DATA TRAINING

Device: cuda:0
Epochs: 3
Batch size: 64

✓ Fresh final model created from SSL encoder

FINAL MODEL — EPOCH 1/3

Training triplets: 101,446
Pseudo triplets:   96,140
Gold triplets:     5,306

Unique studies:
4407
Training batches: 1585


Final Epoch 1:   0%|          | 0/1585 [00:00<?, ?it/s]


Train loss: 0.55071
Time: 19.0 min
Encoder LR: 0.0001000
Head LR: 0.0005000

✓ Checkpoint saved:
/kaggle/working/soft_supervised_gpu/final_full_data_model/final_model_epoch_1.pt

FINAL MODEL — EPOCH 2/3

Training triplets: 101,446
Pseudo triplets:   96,140
Gold triplets:     5,306

Unique studies:
4407
Training batches: 1585


Final Epoch 2:   0%|          | 0/1585 [00:00<?, ?it/s]


Train loss: 0.52952
Time: 19.6 min
Encoder LR: 0.0000934
Head LR: 0.0004666

✓ Checkpoint saved:
/kaggle/working/soft_supervised_gpu/final_full_data_model/final_model_epoch_2.pt

FINAL MODEL — EPOCH 3/3

Training triplets: 101,446
Pseudo triplets:   96,140
Gold triplets:     5,306

Unique studies:
4407
Training batches: 1585


Final Epoch 3:   0%|          | 0/1585 [00:00<?, ?it/s]


Train loss: 0.51825
Time: 19.5 min
Encoder LR: 0.0000752
Head LR: 0.0003753

✓ Checkpoint saved:
/kaggle/working/soft_supervised_gpu/final_full_data_model/final_model_epoch_3.pt

FINAL FULL-DATA TRAINING COMPLETE

Training history:


,epoch,train_loss,time_minutes,encoder_lr,head_lr,training_triplets,pseudo_triplets,gold_triplets
0,1,0.550706,19.039938,0.000100,0.000500,101446,96140,5306
1,2,0.529525,19.564646,0.000093,0.000467,101446,96140,5306
2,3,0.518247,19.510051,0.000075,0.000375,101446,96140,5306



Default final checkpoint:
/kaggle/working/soft_supervised_gpu/final_full_data_model/final_model_epoch_3.pt

✓ All 4,407 studies used
✓ All 58 gold studies used
✓ All 4,349 pseudo studies used
✓ Continuous soft targets preserved
✓ Original validated weighting preserved
✓ TripletMean strategy preserved
✓ Epoch 1/2/3 checkpoints all preserved
✓ NEXT = TEST MRI PREPARATION + INFERENCE


In [7]:
# ============================================================
# STEP 14A-0 — VERIFY OFFICIAL TEST DATA STRUCTURE
# ============================================================

from pathlib import Path
import pandas as pd

COMP_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

TEST_SERIES_ROOT = COMP_ROOT / "test_series"
TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
TEST_CSV = COMP_ROOT / "test.csv"
SAMPLE_SUB_PATH = COMP_ROOT / "sample_submission.csv"

print("=" * 80)
print("STEP 14A-0 — TEST DATA STRUCTURE CHECK")
print("=" * 80)

print("\nFiles:")
print("test.csv exists:            ", TEST_CSV.exists())
print("test_series.csv exists:     ", TEST_SERIES_CSV.exists())
print("test_series folder exists:  ", TEST_SERIES_ROOT.exists())
print("sample_submission exists:   ", SAMPLE_SUB_PATH.exists())

assert TEST_SERIES_CSV.exists()
assert TEST_SERIES_ROOT.exists()
assert SAMPLE_SUB_PATH.exists()

# ------------------------------------------------------------
# Load tables
# ------------------------------------------------------------

test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("\n" + "=" * 80)
print("TEST.CSV")
print("=" * 80)

print("Shape:", test_df.shape)
print("Columns:")
print(test_df.columns.tolist())

print("\nFirst rows:")
display(test_df.head())


print("\n" + "=" * 80)
print("TEST_SERIES.CSV")
print("=" * 80)

print("Shape:", test_series_df.shape)
print("Columns:")
print(test_series_df.columns.tolist())

print(
    "\nUnique studies:",
    test_series_df["StudyInstanceUID"].nunique()
)

print(
    "Unique series:",
    test_series_df["SeriesInstanceUID"].nunique()
)

if "Anatomical_Plane" in test_series_df.columns:
    print("\nSeries by plane:")
    print(
        test_series_df["Anatomical_Plane"]
        .value_counts(dropna=False)
    )

display(test_series_df.head())


print("\n" + "=" * 80)
print("SAMPLE SUBMISSION")
print("=" * 80)

print("Shape:", sample_sub.shape)
print("Columns:")
print(sample_sub.columns.tolist())

display(sample_sub.head())


# ------------------------------------------------------------
# Verify actual DICOM folder structure using first test series
# ------------------------------------------------------------

example = test_series_df.iloc[0]

study_uid = str(example["StudyInstanceUID"])
series_uid = str(example["SeriesInstanceUID"])

example_series_path = (
    TEST_SERIES_ROOT /
    study_uid /
    series_uid
)

print("\n" + "=" * 80)
print("DICOM PATH CHECK")
print("=" * 80)

print("Example study:")
print(study_uid)

print("\nExample series:")
print(series_uid)

print("\nExpected folder:")
print(example_series_path)

print("\nFolder exists:")
print(example_series_path.exists())

if example_series_path.exists():

    dcm_files = sorted(
        example_series_path.glob("*.dcm")
    )

    print("\nNumber of DICOM files:")
    print(len(dcm_files))

    print("\nFirst 5 DICOM files:")

    for p in dcm_files[:5]:
        print(p.name)

else:

    print("\nWARNING: expected folder structure not found.")


# ------------------------------------------------------------
# Coverage comparison
# ------------------------------------------------------------

test_studies_table = set(
    test_series_df["StudyInstanceUID"].astype(str)
)

sample_studies = set(
    sample_sub["StudyInstanceUID"].astype(str)
)

print("\n" + "=" * 80)
print("STUDY COVERAGE CHECK")
print("=" * 80)

print(
    "Studies in test_series.csv:",
    len(test_studies_table)
)

print(
    "Studies in sample_submission:",
    len(sample_studies)
)

print(
    "Missing from sample submission:",
    len(test_studies_table - sample_studies)
)

print(
    "Missing from test_series:",
    len(sample_studies - test_studies_table)
)

print("\n" + "=" * 80)
print("✓ TEST STRUCTURE INSPECTION COMPLETE")
print("NEXT = BUILD NEW TEST SLICE INDEX + 2.5D TRIPLETS")
print("=" * 80)

STEP 14A-0 — TEST DATA STRUCTURE CHECK

Files:
test.csv exists:             True
test_series.csv exists:      True
test_series folder exists:   True
sample_submission exists:    True

TEST.CSV
Shape: (3, 1)
Columns:
['StudyInstanceUID']

First rows:


,StudyInstanceUID
0,1.2.826.0.1.3680043.8.498.10047035057544427318...
1,1.2.826.0.1.3680043.8.498.10062861783145312629...
2,1.2.826.0.1.3680043.8.498.10067514707072572280...



TEST_SERIES.CSV
Shape: (15, 5)
Columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']

Unique studies: 3
Unique series: 15

Series by plane:
Anatomical_Plane
Sagittal    7
Axial       4
Coronal     4
Name: count, dtype: int64


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.11580656442259111255...,0,0,Axial
1,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.17811502614030631664...,0,0,Sagittal
2,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.30565395595045942404...,0,0,Sagittal
3,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.32856494541816845805...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.44334485654554877495...,1,1,Axial



SAMPLE SUBMISSION
Shape: (3, 13)
Columns:
['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5



DICOM PATH CHECK
Example study:
1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191

Example series:
1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947

Expected folder:
/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191/1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947

Folder exists:
True

Number of DICOM files:
34

First 5 DICOM files:
1.2.826.0.1.3680043.8.498.10492923471392639089206565125595901837.dcm
1.2.826.0.1.3680043.8.498.10813088847157507017654677978881920168.dcm
1.2.826.0.1.3680043.8.498.10823532752386915456266439830023207944.dcm
1.2.826.0.1.3680043.8.498.10900781597370949799648805888840803105.dcm
1.2.826.0.1.3680043.8.498.11832386951026218432383619617457640465.dcm

STUDY COVERAGE CHECK
Studies in test_series.csv: 3
Studies in sample_submission: 3
Missing from sample submission: 0
Missing from test_series: 0

✓ TEST STRUCTURE INSPECTION COMPLETE
NE

In [8]:
# ============================================================
# STEP 14A-1 — BUILD TEST SLICE INDEX + 2.5D TRIPLETS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pydicom
from tqdm.auto import tqdm

print("=" * 80)
print("STEP 14A-1 — BUILD TEST SLICE INDEX + 2.5D TRIPLETS")
print("=" * 80)

TEST_WORK = Path(
    "/kaggle/working/soft_supervised_gpu/new_test_preprocessing"
)
TEST_WORK.mkdir(parents=True, exist_ok=True)

TEST_SLICE_INDEX_PATH = TEST_WORK / "test_slice_index.parquet"
TEST_TRIPLETS_PATH = TEST_WORK / "test_triplets.parquet"


# ------------------------------------------------------------
# Helper: slice position
# ------------------------------------------------------------

def get_slice_position(ds):

    if hasattr(ds, "ImagePositionPatient"):
        try:
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)

            if len(ipp) == 3:
                return float(ipp[2]), "ImagePositionPatient"

        except Exception:
            pass

    if hasattr(ds, "SliceLocation"):
        try:
            return float(ds.SliceLocation), "SliceLocation"
        except Exception:
            pass

    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber), "InstanceNumber"
        except Exception:
            pass

    return np.nan, "Unknown"


# ------------------------------------------------------------
# 1. Index DICOM slices
# ------------------------------------------------------------

slice_rows = []
missing_series = []
failed_dicoms = []

for row in tqdm(
    test_series_df.itertuples(index=False),
    total=len(test_series_df),
    desc="Indexing test MRI series"
):

    study_uid = str(row.StudyInstanceUID)
    series_uid = str(row.SeriesInstanceUID)

    series_path = (
        TEST_SERIES_ROOT
        / study_uid
        / series_uid
    )

    if not series_path.exists():

        missing_series.append(series_path)
        continue

    dcm_files = list(series_path.glob("*.dcm"))

    for dcm_path in dcm_files:

        try:

            ds = pydicom.dcmread(
                dcm_path,
                stop_before_pixels=True,
                force=True
            )

            position, position_source = get_slice_position(ds)

            instance_number = (
                float(ds.InstanceNumber)
                if hasattr(ds, "InstanceNumber")
                else np.nan
            )

            rows = (
                int(ds.Rows)
                if hasattr(ds, "Rows")
                else np.nan
            )

            cols = (
                int(ds.Columns)
                if hasattr(ds, "Columns")
                else np.nan
            )

            spacing_y = np.nan
            spacing_x = np.nan

            if hasattr(ds, "PixelSpacing"):

                try:
                    spacing_y = float(ds.PixelSpacing[0])
                    spacing_x = float(ds.PixelSpacing[1])
                except Exception:
                    pass

            slice_rows.append({

                "StudyInstanceUID": study_uid,
                "SeriesInstanceUID": series_uid,

                "Anatomical_Plane":
                    row.Anatomical_Plane,

                "Path":
                    str(dcm_path),

                "Position":
                    position,

                "PositionSource":
                    position_source,

                "InstanceNumber":
                    instance_number,

                "Rows":
                    rows,

                "Columns":
                    cols,

                "PixelSpacingY":
                    spacing_y,

                "PixelSpacingX":
                    spacing_x,
            })

        except Exception as e:

            failed_dicoms.append({
                "Path": str(dcm_path),
                "Error": str(e)
            })


test_slice_df = pd.DataFrame(slice_rows)


# ------------------------------------------------------------
# 2. QC slice index
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST SLICE INDEX QC")
print("=" * 80)

print("Indexed slices:", len(test_slice_df))

print(
    "Unique studies:",
    test_slice_df["StudyInstanceUID"].nunique()
)

print(
    "Unique series:",
    test_slice_df["SeriesInstanceUID"].nunique()
)

print(
    "Missing series folders:",
    len(missing_series)
)

print(
    "Failed DICOMs:",
    len(failed_dicoms)
)

print("\nPosition source counts:")
print(
    test_slice_df["PositionSource"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 3. Sort slices inside every series
# ------------------------------------------------------------

sorted_groups = []

for (
    study_uid,
    series_uid
), group in test_slice_df.groupby(
    [
        "StudyInstanceUID",
        "SeriesInstanceUID"
    ],
    sort=False
):

    group = group.copy()

    # Prefer physical position
    if group["Position"].notna().all():

        group = group.sort_values(
            [
                "Position",
                "InstanceNumber"
            ],
            kind="mergesort"
        )

    else:

        group = group.sort_values(
            "InstanceNumber",
            kind="mergesort"
        )

    sorted_groups.append(
        group.reset_index(drop=True)
    )


test_slice_df = pd.concat(
    sorted_groups,
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Save slice index
# ------------------------------------------------------------

test_slice_df.to_parquet(
    TEST_SLICE_INDEX_PATH,
    index=False
)

print("\nSaved:")
print(TEST_SLICE_INDEX_PATH)


# ------------------------------------------------------------
# 5. Build adjacent 2.5D triplets
# ------------------------------------------------------------

triplet_rows = []

grouped = test_slice_df.groupby(
    [
        "StudyInstanceUID",
        "SeriesInstanceUID"
    ],
    sort=False
)

n_series = test_slice_df[
    [
        "StudyInstanceUID",
        "SeriesInstanceUID"
    ]
].drop_duplicates().shape[0]


for (
    study_uid,
    series_uid
), group in tqdm(
    grouped,
    total=n_series,
    desc="Building 2.5D test triplets"
):

    group = group.reset_index(drop=True)

    n = len(group)

    if n < 3:
        continue

    for centre_idx in range(1, n - 1):

        prev_row = group.iloc[centre_idx - 1]
        centre_row = group.iloc[centre_idx]
        next_row = group.iloc[centre_idx + 1]

        triplet_rows.append({

            "StudyInstanceUID":
                study_uid,

            "SeriesInstanceUID":
                series_uid,

            "Anatomical_Plane":
                centre_row["Anatomical_Plane"],

            "PreviousPath":
                prev_row["Path"],

            "CentrePath":
                centre_row["Path"],

            "NextPath":
                next_row["Path"],

            "PreviousPosition":
                prev_row["Position"],

            "CentrePosition":
                centre_row["Position"],

            "NextPosition":
                next_row["Position"],

            "PositionSource":
                centre_row["PositionSource"],

            "InstanceNumber":
                centre_row["InstanceNumber"],

            "Rows":
                centre_row["Rows"],

            "Columns":
                centre_row["Columns"],

            "PixelSpacingY":
                centre_row["PixelSpacingY"],

            "PixelSpacingX":
                centre_row["PixelSpacingX"],

            "CentreIndex":
                centre_idx,
        })


test_triplets_df = pd.DataFrame(triplet_rows)


# ------------------------------------------------------------
# 6. Triplet QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST TRIPLET QC")
print("=" * 80)

print(
    "Triplets:",
    len(test_triplets_df)
)

print(
    "Unique studies:",
    test_triplets_df[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Unique series:",
    test_triplets_df[
        "SeriesInstanceUID"
    ].nunique()
)

print("\nTriplets by plane:")

print(
    test_triplets_df[
        "Anatomical_Plane"
    ].value_counts()
)

print("\nTriplets per series:")

print(
    test_triplets_df
    .groupby(
        "SeriesInstanceUID"
    )
    .size()
    .describe()
)


# ------------------------------------------------------------
# 7. Path QC
# ------------------------------------------------------------

print("\nPath validation:")

for col in [
    "PreviousPath",
    "CentrePath",
    "NextPath"
]:

    missing = (
        ~test_triplets_df[col]
        .map(lambda x: Path(x).exists())
    ).sum()

    print(
        f"{col:<20} missing = {missing}"
    )


# ------------------------------------------------------------
# 8. Critical assertions
# ------------------------------------------------------------

expected_studies = (
    test_series_df[
        "StudyInstanceUID"
    ].nunique()
)

expected_series = (
    test_series_df[
        "SeriesInstanceUID"
    ].nunique()
)


assert len(missing_series) == 0

assert len(failed_dicoms) == 0

assert (
    test_triplets_df[
        "StudyInstanceUID"
    ].nunique()
    == expected_studies
)

assert (
    test_triplets_df[
        "SeriesInstanceUID"
    ].nunique()
    == expected_series
)

assert (
    test_triplets_df[
        [
            "PreviousPath",
            "CentrePath",
            "NextPath"
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

test_triplets_df.to_parquet(
    TEST_TRIPLETS_PATH,
    index=False
)

print("\nSaved:")
print(TEST_TRIPLETS_PATH)


print("\n" + "=" * 80)
print("✓ TEST SLICE INDEX COMPLETE")
print("✓ TEST 2.5D TRIPLETS COMPLETE")
print("NEXT = COMPUTE TEST P1/P99")
print("=" * 80)

STEP 14A-1 — BUILD TEST SLICE INDEX + 2.5D TRIPLETS


Indexing test MRI series:   0%|          | 0/15 [00:00<?, ?it/s]


TEST SLICE INDEX QC
Indexed slices: 557
Unique studies: 3
Unique series: 15
Missing series folders: 0
Failed DICOMs: 0

Position source counts:
PositionSource
ImagePositionPatient    557
Name: count, dtype: int64

Saved:
/kaggle/working/soft_supervised_gpu/new_test_preprocessing/test_slice_index.parquet


Building 2.5D test triplets:   0%|          | 0/15 [00:00<?, ?it/s]


TEST TRIPLET QC
Triplets: 527
Unique studies: 3
Unique series: 15

Triplets by plane:
Anatomical_Plane
Axial       259
Sagittal    166
Coronal     102
Name: count, dtype: int64

Triplets per series:
count     15.000000
mean      35.133333
std       34.321727
min       22.000000
25%       22.000000
50%       28.000000
75%       29.500000
max      158.000000
dtype: float64

Path validation:
PreviousPath         missing = 0
CentrePath           missing = 0
NextPath             missing = 0

Saved:
/kaggle/working/soft_supervised_gpu/new_test_preprocessing/test_triplets.parquet

✓ TEST SLICE INDEX COMPLETE
✓ TEST 2.5D TRIPLETS COMPLETE
NEXT = COMPUTE TEST P1/P99


In [9]:
# ============================================================
# STEP 14B — COMPUTE TEST SERIES P1 / P99
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pydicom
from tqdm.auto import tqdm

print("=" * 80)
print("STEP 14B — COMPUTE TEST P1/P99")
print("=" * 80)

TEST_PERCENTILES_PATH = (
    TEST_WORK /
    "test_series_percentiles.parquet"
)


# ------------------------------------------------------------
# Helper: read DICOM pixels with rescale
# ------------------------------------------------------------

def read_rescaled_pixels(path):

    ds = pydicom.dcmread(
        path,
        force=True
    )

    arr = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    arr = (
        arr * slope
        + intercept
    )

    return arr


# ------------------------------------------------------------
# Compute percentiles per series
# ------------------------------------------------------------

percentile_rows = []

failed_series = []

series_groups = test_slice_df.groupby(
    [
        "StudyInstanceUID",
        "SeriesInstanceUID"
    ],
    sort=False
)

n_series = (
    test_slice_df[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

for (
    study_uid,
    series_uid
), group in tqdm(
    series_groups,
    total=n_series,
    desc="Computing test P1/P99"
):

    sampled_pixels = []

    try:

        for path in group["Path"]:

            arr = read_rescaled_pixels(
                path
            )

            flat = arr.reshape(-1)

            # Keep computation bounded and reproducible
            max_pixels_per_slice = 50000

            if len(flat) > max_pixels_per_slice:

                idx = np.linspace(
                    0,
                    len(flat) - 1,
                    max_pixels_per_slice,
                    dtype=np.int64
                )

                flat = flat[idx]

            sampled_pixels.append(
                flat.astype(
                    np.float32,
                    copy=False
                )
            )

        if len(sampled_pixels) == 0:

            raise RuntimeError(
                "No readable slices."
            )

        all_pixels = np.concatenate(
            sampled_pixels
        )

        finite_mask = np.isfinite(
            all_pixels
        )

        all_pixels = all_pixels[
            finite_mask
        ]

        if len(all_pixels) == 0:

            raise RuntimeError(
                "No finite pixels."
            )

        p1 = float(
            np.percentile(
                all_pixels,
                1
            )
        )

        p99 = float(
            np.percentile(
                all_pixels,
                99
            )
        )

        valid = (
            np.isfinite(p1)
            and np.isfinite(p99)
            and p99 > p1
        )

        percentile_rows.append({

            "StudyInstanceUID":
                study_uid,

            "SeriesInstanceUID":
                series_uid,

            "P1":
                p1,

            "P99":
                p99,

            "Valid":
                bool(valid),

            "NumSlices":
                len(group),

            "NumSampledPixels":
                len(all_pixels),
        })

    except Exception as e:

        failed_series.append({

            "StudyInstanceUID":
                study_uid,

            "SeriesInstanceUID":
                series_uid,

            "Error":
                str(e)
        })


test_percentiles_df = pd.DataFrame(
    percentile_rows
)


# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEST P1/P99 QC")
print("=" * 80)

print(
    "Rows:",
    len(test_percentiles_df)
)

print(
    "Unique studies:",
    test_percentiles_df[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Unique series:",
    test_percentiles_df[
        "SeriesInstanceUID"
    ].nunique()
)

print(
    "Failed series:",
    len(failed_series)
)

print("\nValid counts:")

print(
    test_percentiles_df[
        "Valid"
    ].value_counts(
        dropna=False
    )
)

print("\nP1 summary:")

print(
    test_percentiles_df[
        "P1"
    ].describe()
)

print("\nP99 summary:")

print(
    test_percentiles_df[
        "P99"
    ].describe()
)


# ------------------------------------------------------------
# Merge into triplet manifest
# ------------------------------------------------------------

test_triplets_ready = (
    test_triplets_df
    .merge(
        test_percentiles_df[
            [
                "SeriesInstanceUID",
                "P1",
                "P99",
                "Valid"
            ]
        ],
        on="SeriesInstanceUID",
        how="left"
    )
)


print("\n" + "=" * 80)
print("MERGED TEST MANIFEST QC")
print("=" * 80)

print(
    "Triplets:",
    len(test_triplets_ready)
)

print(
    "Missing P1:",
    test_triplets_ready[
        "P1"
    ].isna().sum()
)

print(
    "Missing P99:",
    test_triplets_ready[
        "P99"
    ].isna().sum()
)

print(
    "Invalid percentile triplets:",
    (
        test_triplets_ready[
            "Valid"
        ] != True
    ).sum()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(
    test_percentiles_df
) == 15

assert len(
    failed_series
) == 0

assert (
    test_percentiles_df[
        "Valid"
    ].all()
)

assert (
    test_triplets_ready[
        [
            "P1",
            "P99"
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

test_percentiles_df.to_parquet(
    TEST_PERCENTILES_PATH,
    index=False
)

TEST_READY_PATH = (
    TEST_WORK /
    "test_triplets_ready.parquet"
)

test_triplets_ready.to_parquet(
    TEST_READY_PATH,
    index=False
)


print("\nSaved percentiles:")
print(TEST_PERCENTILES_PATH)

print("\nSaved inference-ready manifest:")
print(TEST_READY_PATH)


print("\n" + "=" * 80)
print("✓ TEST P1/P99 COMPLETE")
print("✓ TEST PREPROCESSING COMPLETE")
print("NEXT = QUICK SAVE THIS NOTEBOOK")
print("THEN SWITCH TO FINAL SUBMISSION NOTEBOOK")
print("=" * 80)

STEP 14B — COMPUTE TEST P1/P99


Computing test P1/P99:   0%|          | 0/15 [00:00<?, ?it/s]


TEST P1/P99 QC
Rows: 15
Unique studies: 3
Unique series: 15
Failed series: 0

Valid counts:
Valid
True    15
Name: count, dtype: int64

P1 summary:
count    15.000000
mean     -0.066667
std       2.463060
min      -8.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       4.000000
Name: P1, dtype: float64

P99 summary:
count      15.000000
mean     2236.666667
std      1998.364462
min       219.000000
25%       830.500000
50%      1817.000000
75%      3137.000000
max      7533.000000
Name: P99, dtype: float64

MERGED TEST MANIFEST QC
Triplets: 527
Missing P1: 0
Missing P99: 0
Invalid percentile triplets: 0

Saved percentiles:
/kaggle/working/soft_supervised_gpu/new_test_preprocessing/test_series_percentiles.parquet

Saved inference-ready manifest:
/kaggle/working/soft_supervised_gpu/new_test_preprocessing/test_triplets_ready.parquet

✓ TEST P1/P99 COMPLETE
✓ TEST PREPROCESSING COMPLETE
NEXT = QUICK SAVE THIS NOTEBOOK
THEN SWITCH TO FINAL SUBMISSION NOTEBOOK
